# End-to-End Quant Factor Workflow with Snowflake ML

**Objective**: Build a next-generation factor strategy — from raw data to deployed pipeline — using Snowflake ML capabilities.

**Demo Flow** (9 steps across 4 phases):
1. **Factor Construction with Feature Store** — 3 domain-specific FeatureViews aligned with source cadences: market factors (daily, 5 features), fundamental factors (60 days, 8 features from Cybersyn exposures + financial statements + dividends), and sentiment factors (weekly, 5 NLP dimensions via AI_SENTIMENT)
     * **NLP Sentiment from Earnings Transcripts** — 5 sentiment dimensions via `ai_agg()` with categories (guidance, margins, growth, risk). Tests whether alternative data adds marginal alpha beyond traditional quant factors.
     * **Derived Features** — 4 interaction features (momentum/value ratio, quality×growth, sentiment×momentum, factor dispersion) + forward return target as separate FeatureViews
     * **Hidden Thematic Factor Construction** — 5 non-traditional factors (AI_Exposure, Reshoring_Benefit, Rate_Convexity, Climate_Transition, Geopolitical_Risk) from SEC filings, earnings transcripts, and ESG data. Validated via coverage, IC, and factor correlation gates.
2. **Cross-Sectional Factor Model (Fama-MacBeth)** — Econometric validation: are these 11 factors (6 traditional + 5 hidden) priced by the market? Produces factor return premia and t-statistics
3. **ML Factor Discovery (XGBoost + SHAP)** — Builds on Phase 2: which of the 27 features (13 quant + 5 NLP + 5 hidden + 4 derived) best predict individual stock returns? 3-way SHAP comparison (Quant → +NLP → +Hidden) measures incremental alpha.
4. **Model Registration** — Versioned model with IC and turnover metrics
5. **Portfolio Construction** — Mean-variance optimisation with ML-predicted returns
6. **ML Observability** — Model Monitor for factor drift
7. **Pipeline Deployment** — DAG API for end-to-end automation

**Source Data**: Daily stock prices and security dimensions for the full universe (~100 tickers across sectors). Using all stocks ensures sufficient cross-sectional breadth for Fama-MacBeth regressions (which need many stocks per month) and gives XGBoost enough variation to learn factor-return relationships across different market caps, sectors, and styles.

**Additional Sources**: Daily security returns, factor exposures, SEC financial statements, benchmark returns, dividend payments, regime predictions, earnings call transcripts, SEC segment revenue, transcript NLP scores, ESG scores

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from datetime import datetime

from snowflake.snowpark import functions as F
from snowflake.snowpark import types as T
from snowflake.snowpark import Window as W

try:
    from snowflake.snowpark.context import get_active_session
    session = get_active_session()
except:
    from snowflake.snowpark import Session
    session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "sfseeurope-mstellwall-aws-us-west3")).create()

DATABASE = "SAM_DEMO"
ML_SCHEMA = "ML"
CURATED = "CURATED"
MARKET_DATA = "MARKET_DATA"

session.sql(f"USE DATABASE {DATABASE}").collect()
session.sql(f"USE SCHEMA {ML_SCHEMA}").collect()
print(f"Connected: {session.get_current_account()} | {DATABASE}.{ML_SCHEMA}")

/Users/mstellwall/anaconda3/envs/sam-demo/lib/python3.11/site-packages/snowflake/connector/vendored/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(
/Users/mstellwall/anaconda3/envs/sam-demo/lib/python3.11/site-packages/requests/__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


Connected: "sfseeurope-mstellwall-aws-us-west3" | SAM_DEMO.ML


## Phase 1: Factor Construction with Feature Store

We construct 13 quant factors from daily stock prices, factor exposures, and fundamentals for the full stock universe, then register them as FeatureViews that auto-create managed Dynamic Tables. Six factors (Value, Size, Quality, Growth, Volatility, Market) arrive pre-z-scored from the production exposure pipeline; the remaining seven (momentum, liquidity, volume trend, leverage, profitability, earnings revision, dividend yield) are z-scored from raw data with ±3σ winsorisation.

### Source Data and Monthly Aggregation

Load daily stock prices (filtered to the last 5 years), security dimensions, returns, and factor exposures. Join prices with security metadata, truncate to monthly granularity, and aggregate to one row per ticker per month.

In [2]:
prices = session.table(f"{DATABASE}.{MARKET_DATA}.FACT_STOCK_PRICES").filter(F.col("PRICE_DATE") >= F.dateadd("year", F.lit(-5), F.current_date()))
dim_sec = session.table(f"{DATABASE}.{CURATED}.DIM_SECURITY")

mp = (prices
        .join(dim_sec, prices["SECURITYID"] == dim_sec["SECURITYID"], "inner")
        .select(dim_sec["TICKER"].alias("TICKER")
        , F.call_builtin("DATE_TRUNC", F.lit("month"), prices["PRICE_DATE"]).alias("MONTH_DATE")
        , prices["PRICE_DATE"]
        , prices["PRICE_CLOSE"]
        , prices["VOLUME"]
        , prices["SecurityID"].alias("SEC_ID"),)
    )

eom_window = W.partition_by("TICKER", "MONTH_DATE").order_by(F.col("PRICE_DATE").desc())
mp_ranked = mp.with_column("_EOM_RANK", F.row_number().over(eom_window))

mp_agg = (mp_ranked
            .group_by("TICKER", "MONTH_DATE")
            .agg(F.max(F.when(F.col("_EOM_RANK") == 1, F.col("PRICE_CLOSE"))).alias("CLOSE_PRICE")
                        , F.sum("VOLUME").alias("TOTAL_VOLUME")
                        , F.avg("VOLUME").alias("AVG_VOLUME")
                        , F.max("SEC_ID").alias("SEC_ID"),)
        )

print(f"Monthly price records: {mp_agg.count()}")
mp_agg.show(5)

Monthly price records: 29882
----------------------------------------------------------------------------------------
|"TICKER"  |"MONTH_DATE"  |"CLOSE_PRICE"  |"TOTAL_VOLUME"  |"AVG_VOLUME"    |"SEC_ID"  |
----------------------------------------------------------------------------------------
|BTI       |2023-02-01    |38.06          |9587110         |504584.736842   |124       |
|STNE      |2023-11-01    |15.55          |41847514        |1992738.761905  |426       |
|QRVO      |2023-08-01    |107.4          |14301012        |621783.130435   |373       |
|FLEX      |2023-07-01    |27.5           |21206117        |1060305.850000  |209       |
|GLD       |2026-04-01    |434.0          |19245246        |1283016.400000  |420       |
----------------------------------------------------------------------------------------



#### Data Quality Gate: Universe Stability and Completeness

Before building any factors, we verify that the raw price universe is stable and complete. Cross-sectional models like Fama-MacBeth require a consistent universe — if tickers appear and disappear unpredictably, factor regressions become unreliable because the cross-section changes composition each month.

**What to look for:**
- **Universe size over time** — Should be roughly stable. Sudden drops below ~30 tickers mean insufficient cross-sectional breadth for Fama-MacBeth regressions in those months.
- **Missing close prices** — Should be rare (<5% of observations). Widespread NULLs propagate into momentum, volume, and all downstream features.
- **Coverage gaps** — Months with zero data indicate a source outage and must be excluded from training.

**Decision impact:** Months with fewer than 30 tickers are excluded from Fama-MacBeth (Phase 2) and flagged in XGBoost training (Phase 3). If >10% of observations have missing prices, investigate the data source before proceeding — all downstream factors inherit this noise.

In [3]:
coverage_sf = (mp_agg.group_by("MONTH_DATE").agg(
    F.count_distinct("TICKER").alias("N_TICKERS"),
    F.count("*").alias("N_OBS"),
    F.sum(F.when(F.col("CLOSE_PRICE").is_null(), 1).otherwise(0)).alias("N_MISSING")
).with_column("MISSING_PCT", F.col("N_MISSING") / F.col("N_OBS") * 100))

verdicts = coverage_sf.select(
    F.min("N_TICKERS").alias("MIN_TICKERS"),
    F.avg("N_TICKERS").alias("AVG_TICKERS"),
    F.max("MISSING_PCT").alias("MAX_MISSING"),
    F.sum(F.when(F.col("N_TICKERS") < 30, 1).otherwise(0)).alias("LOW_MONTHS")
).to_pandas().iloc[0]

coverage = coverage_sf.sort("MONTH_DATE").to_pandas()

fig = make_subplots(rows=1, cols=2, subplot_titles=("Universe Size Over Time", "Missing Close Prices (%)"))

fig.add_trace(go.Scatter(x=coverage["MONTH_DATE"], y=coverage["N_TICKERS"],
    mode="lines+markers", marker=dict(size=3), name="Tickers"), row=1, col=1)
fig.add_hline(y=30, line_dash="dash", line_color="red", opacity=0.5,
    annotation_text="Min threshold (30)", row=1, col=1)

fig.add_trace(go.Bar(x=coverage["MONTH_DATE"], y=coverage["MISSING_PCT"],
    opacity=0.7, name="Missing %"), row=1, col=2)
fig.add_hline(y=5, line_dash="dash", line_color="red", opacity=0.5,
    annotation_text="5% threshold", row=1, col=2)

fig.update_yaxes(title_text="Distinct Tickers", row=1, col=1)
fig.update_yaxes(title_text="% Missing", row=1, col=2)
fig.update_layout(height=350, template="plotly_white", showlegend=False)
fig.show()

min_tickers = int(verdicts["MIN_TICKERS"])
avg_tickers = float(verdicts["AVG_TICKERS"])
max_missing = float(verdicts["MAX_MISSING"])
low_months = int(verdicts["LOW_MONTHS"])

print(f"{'Metric':<30} {'Value':>10} {'Status':>10}")
print("-" * 55)
print(f"{'Min tickers in any month':<30} {min_tickers:>10} {'FAIL' if min_tickers < 30 else 'PASS':>10}")
print(f"{'Avg tickers per month':<30} {avg_tickers:>10.0f} {'PASS':>10}")
print(f"{'Max missing close %':<30} {max_missing:>9.1f}% {'FAIL' if max_missing > 5 else 'PASS':>10}")
print(f"{'Months below 30 tickers':<30} {low_months:>10} {'REVIEW' if low_months > 0 else 'PASS':>10}")

Metric                              Value     Status
-------------------------------------------------------
Min tickers in any month              439       PASS
Avg tickers per month                 490       PASS
Max missing close %                  0.0%       PASS
Months below 30 tickers                 0       PASS


### Momentum Features via Lag Computation

Compute 1-month, 3-month, and 12-month price lags per ticker, then derive momentum as percentage price change.   
**Momentum** is one of the most robust cross-sectional predictors of future returns (Jegadeesh & Titman, 1993) — stocks that outperformed recently tend to continue outperforming over the near term. The 1-month lag captures short-term continuation, while 3- and 12-month lags capture medium- and long-term trend persistence. 

These feed Phase 2's Fama-MacBeth regressions (to test whether momentum earns a significant premium) and Phase 3's XGBoost model (as a core feature for predicting individual stock returns).

In [4]:
mp_with_lags = (mp_agg
    .analytics.compute_lag(
        cols=["CLOSE_PRICE", "TOTAL_VOLUME"]
        , lags=[1, 3, 12]
        , order_by=["MONTH_DATE"]
        , group_by=["TICKER"]
        , col_formatter=lambda col, op, val: f"{col}_LAG_{val}"
    )
)

price_features = (mp_with_lags
    .with_column("MOMENTUM_1M"
        , (F.col("CLOSE_PRICE") / F.when(F.col("CLOSE_PRICE_LAG_1") != 0, F.col("CLOSE_PRICE_LAG_1")).otherwise(F.lit(None))) - F.lit(1))
    .with_column("MOMENTUM_3M"
        , (F.col("CLOSE_PRICE") / F.when(F.col("CLOSE_PRICE_LAG_3") != 0, F.col("CLOSE_PRICE_LAG_3")).otherwise(F.lit(None))) - F.lit(1))
    .with_column("MOMENTUM_12M"
        , (F.col("CLOSE_PRICE") / F.when(F.col("CLOSE_PRICE_LAG_12") != 0, F.col("CLOSE_PRICE_LAG_12")).otherwise(F.lit(None))) - F.lit(1))
    .with_column("MARKET_CAP_PROXY", F.col("CLOSE_PRICE") * F.col("AVG_VOLUME"))
    .with_column("VOLUME_TREND"
        , (F.col("TOTAL_VOLUME") / F.when(F.col("TOTAL_VOLUME_LAG_1") != 0, F.col("TOTAL_VOLUME_LAG_1")).otherwise(F.lit(None))) - F.lit(1))
    .select("TICKER", "MONTH_DATE", "CLOSE_PRICE", "AVG_VOLUME", "TOTAL_VOLUME", "SEC_ID"
        , "MOMENTUM_1M", "MOMENTUM_3M", "MOMENTUM_12M", "MARKET_CAP_PROXY", "VOLUME_TREND"
    )
)

print(f"Price features: {price_features.count()} rows")
price_features.show(5)

Price features: 29882 rows
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"TICKER"  |"MONTH_DATE"  |"CLOSE_PRICE"  |"AVG_VOLUME"    |"TOTAL_VOLUME"  |"SEC_ID"  |"MOMENTUM_1M"         |"MOMENTUM_3M"         |"MOMENTUM_12M"  |"MARKET_CAP_PROXY"  |"VOLUME_TREND"  |
---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|KDP       |2021-04-01    |35.55          |1981378.200000  |9906891         |274       |NULL                  |NULL                  |NULL            |70437995.00999999   |NULL            |
|KDP       |2021-05-01    |36.96          |1537617.100000  |30752342        |274       |0.039662447257384104  |NULL                  |NULL            |56830328.016        |2.104137        |
|KDP       |2021-06-01 

#### Factor Quality Gate: Momentum Signal Validation

Momentum (past winners continue to win) is one of the most documented cross-sectional predictors of future returns, but it can break down in crash regimes or small universes. Before carrying momentum features forward, we check three properties:

1. **Distribution shape** — Momentum returns should be roughly symmetric with moderate tails. Heavy skewness suggests survivorship bias or outlier contamination that would distort z-scores downstream.
2. **Univariate rank IC** — Does higher momentum predict higher next-month returns? We compute Spearman rank correlation between momentum and 1-month forward return for each cross-section. A mean |IC| > 0.02 is encouraging; < 0.01 suggests the signal is too noisy for this universe.
3. **Autocorrelation of ranks** — Momentum should persist for 1-6 months then decay. If cross-sectional rank autocorrelation is zero at lag 1, there is no persistence and the signal is likely noise.

**Decision impact:** If all three momentum horizons (1M, 3M, 12M) show |IC| < 0.01, momentum may not be useful for this universe — but we still carry it forward as XGBoost can find non-linear interactions. Low autocorrelation means the factor needs monthly recomputation (no forward-filling). Severe skewness (|skew| > 2) warrants winsorisation before z-scoring.

In [15]:
from snowflake.snowpark.functions import corr as sf_corr, rank as sf_rank, skew as sf_skew

mom_sf = (price_features
    .select("TICKER", "MONTH_DATE", "CLOSE_PRICE", "MOMENTUM_1M", "MOMENTUM_3M", "MOMENTUM_12M")
    .with_column("FWD_RETURN_1M",
        F.lead("CLOSE_PRICE", 1).over(W.partition_by("TICKER").order_by("MONTH_DATE"))
        / F.col("CLOSE_PRICE") - 1)
    .dropna()
)

mom_cols = ["MOMENTUM_1M", "MOMENTUM_3M", "MOMENTUM_12M"]

mom_stats = mom_sf.select(
    *[F.avg(c).alias(f"{c}_MEAN") for c in mom_cols],
    *[F.stddev(F.col(c)).alias(f"{c}_STD") for c in mom_cols],
    *[sf_skew(F.col(c)).alias(f"{c}_SKEW") for c in mom_cols],
).to_pandas().iloc[0]

mom_hist_pd = mom_sf.select(*mom_cols).to_pandas()

mom_ranked = (mom_sf
                .with_columns([f"RANK_{c}" for c in mom_cols]
                            , [sf_rank().over(W.partition_by("MONTH_DATE").order_by(c)) for c in mom_cols])
                .with_column("RANK_FWD", sf_rank().over(W.partition_by("MONTH_DATE").order_by("FWD_RETURN_1M")))
                .with_column("N", F.count("*").over(W.partition_by("MONTH_DATE")))
                .filter(F.col("N") >= 10)
            )

all_ics = (mom_ranked
    .group_by("MONTH_DATE")
    .agg(*[sf_corr(F.col(f"RANK_{c}"), F.col("RANK_FWD")).alias(f"IC_{c}") for c in mom_cols])
    .sort("MONTH_DATE")
).to_pandas()

all_ics_long = all_ics.melt(id_vars=["MONTH_DATE"], value_vars=[f"IC_{c}" for c in mom_cols],
    var_name="FACTOR", value_name="IC")
all_ics_long["FACTOR"] = all_ics_long["FACTOR"].str.replace("IC_", "")

fig = make_subplots(rows=2, cols=3, subplot_titles=["" for _ in range(6)])

for i, col_name in enumerate(mom_cols):
    vals = mom_hist_pd[col_name]
    fig.add_trace(go.Histogram(x=vals.clip(-1, 1), nbinsx=50, opacity=0.7,
        name=col_name, showlegend=False), row=1, col=i+1)
    fig.add_vline(x=0, line_dash="dash", line_color="red", opacity=0.5, row=1, col=i+1)
    m, s, sk = mom_stats[f"{col_name}_MEAN"], mom_stats[f"{col_name}_STD"], mom_stats[f"{col_name}_SKEW"]
    fig.update_xaxes(title_text=f"{col_name}<br>mean={m:.3f} std={s:.3f} skew={sk:.2f}", row=1, col=i+1)

for i, col_name in enumerate(mom_cols):
    ic_vals = all_ics_long.loc[all_ics_long["FACTOR"] == col_name, "IC"].dropna()
    mean_ic = ic_vals.mean() if len(ic_vals) > 0 else 0
    fig.add_trace(go.Histogram(x=ic_vals, nbinsx=30, opacity=0.7,
        name=f"IC {col_name}", showlegend=False), row=2, col=i+1)
    fig.add_vline(x=mean_ic, line_color="red", line_width=2, row=2, col=i+1)
    fig.add_vline(x=0, line_dash="dash", line_color="black", opacity=0.3, row=2, col=i+1)
    fig.update_xaxes(title_text=f"{col_name} — Monthly Rank IC (Mean={mean_ic:.4f})", row=2, col=i+1)

fig.update_layout(height=600, template="plotly_white",
    title_text="Momentum Factor Quality Checks", title_font_size=14)
fig.show()

print(f"\n{'Factor':<20} {'Mean IC':>10} {'|IC| > 0.02':>12} {'Skewness':>10} {'Status':>10}")
print("-" * 65)
for col_name in mom_cols:
    ic_vals = all_ics_long.loc[all_ics_long["FACTOR"] == col_name, "IC"].dropna()
    mean_ic = ic_vals.mean() if len(ic_vals) > 0 else 0
    skew_val = float(mom_stats[f"{col_name}_SKEW"])
    status = "PASS" if abs(mean_ic) > 0.02 else "REVIEW" if abs(mean_ic) > 0.01 else "WEAK"
    print(f"{col_name:<20} {mean_ic:>+10.4f} {'Yes' if abs(mean_ic) > 0.02 else 'No':>12} {skew_val:>10.2f} {status:>10}")


Factor                  Mean IC  |IC| > 0.02   Skewness     Status
-----------------------------------------------------------------
MOMENTUM_1M             +0.0159           No      63.10     REVIEW
MOMENTUM_3M             -0.0084           No      37.44       WEAK
MOMENTUM_12M            +0.0196           No      66.47     REVIEW


### Factor Exposure Joins

Pivot each of the six pre-computed factors from `FACT_FACTOR_EXPOSURES` (long-format) into separate columns, then left-join them onto the monthly price features by security ID and month. These exposures are already z-scored and winsorised (±3σ) by the production data pipeline.

| Factor | Column | What it measures |
|--------|--------|-----------------|
| **Value** | `VALUE_EXPOSURE` | Composite earnings yield: average of EPS / price (earnings yield) and book value / market cap (book-to-market ratio) |
| **Size** | `SIZE_EXPOSURE` | Natural log of market capitalisation (shares outstanding × close price) |
| **Quality** | `QUALITY_EXPOSURE` | Composite profitability metric: (ROE + operating margin − debt-to-equity) / 3 |
| **Growth** | `GROWTH_EXPOSURE` | Revenue growth rate (year-over-year revenue growth percentage / 100) |
| **Volatility** | `VOLATILITY_EXPOSURE` | 60-day rolling standard deviation of daily returns (minimum 30 observations) |
| **Market (Beta)** | `MARKET_EXPOSURE` | Regression slope (REGR_SLOPE) of security daily returns vs SPX over a 252-day rolling window (minimum 120 observations) |

> **Note:** The production pipeline also computes a **Momentum** exposure (12-1 month return, skipping the most recent month), but the notebook excludes it in favour of its own 1-month momentum calculated from price lags above.

In [10]:
fact_exp = session.table(f"{DATABASE}.{CURATED}.FACT_FACTOR_EXPOSURES")

exposures = (fact_exp
        .with_column("MONTH_DATE", F.call_builtin("DATE_TRUNC", F.lit("month"), F.col("EXPOSURE_DATE")))
        .pivot(F.col("FACTOR_NAME"), ["Value", "Size", "Quality", "Growth", "Volatility", "Market"])
        .agg(F.max(F.col("EXPOSURE_VALUE")))
        .select(
            F.col("SecurityID").alias("EXP_SID")
            , F.col("MONTH_DATE").alias("EXP_MONTH")
            , F.col("'Value'").alias("VALUE_EXPOSURE")
            , F.col("'Size'").alias("SIZE_EXPOSURE")
            , F.col("'Quality'").alias("QUALITY_EXPOSURE")
            , F.col("'Growth'").alias("GROWTH_EXPOSURE")
            , F.col("'Volatility'").alias("VOLATILITY_EXPOSURE")
            , F.col("'Market'").alias("MARKET_EXPOSURE")
        )
    )

raw = (price_features
        .join(exposures, (price_features["SEC_ID"] == exposures["EXP_SID"]) & (price_features["MONTH_DATE"] == exposures["EXP_MONTH"]), "left")
        .select("TICKER", "MONTH_DATE", "SEC_ID"
            , "MOMENTUM_1M", "MOMENTUM_3M", "MOMENTUM_12M"
            , "AVG_VOLUME", "MARKET_CAP_PROXY", "VOLUME_TREND"
            , "VALUE_EXPOSURE", "SIZE_EXPOSURE", "QUALITY_EXPOSURE", "GROWTH_EXPOSURE"
            , "VOLATILITY_EXPOSURE", "MARKET_EXPOSURE"
        )
    )

#print(f"Raw features with exposures: {raw.count()} rows")
#print(f"VOLATILITY_EXPOSURE nulls: {raw.filter(F.col('VOLATILITY_EXPOSURE').is_null()).count()}")
#print(f"MARKET_EXPOSURE nulls: {raw.filter(F.col('MARKET_EXPOSURE').is_null()).count()}")
#raw.show(5)
raw.describe().show()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|"SUMMARY"  |"TICKER"  |"SEC_ID"            |"MOMENTUM_1M"         |"MOMENTUM_3M"         |"MOMENTUM_12M"       |"AVG_VOLUME"        |"MARKET_CAP_PROXY"  |"VOLUME_TREND"     |"VALUE_EXPOSURE"     |"SIZE_EXPOSURE"       |"QUALITY_EXPOSURE"   |"GROWTH_EXPOSURE"       |"VOLATILITY_EXPOSURE"  |"MARKET_EXPOSURE"     |
--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
|stddev     |NULL      |145.88728446646746  |0.37859126

### Raw Factor Computation

Compute four additional factors from raw financial and market data (these are z-scored later alongside the pre-computed exposures):
- **Leverage** — debt-to-equity from annual balance sheet filings
- **Profitability** — ROE percentage scaled to decimal (`ROE_PCT / 100`)
- **Earnings Revision** — quarter-over-quarter EPS change
- **Dividend Yield** — trailing 12-month dividends per share / close price

#### Leverage and Profitability — from SEC financial statements

Source: SEC financial statements (joined to security dimensions via issuer).
- **Leverage**: Debt-to-equity from annual/TTM balance sheet filings (`COVERED_QTRS=0`). Measures financial risk — highly leveraged firms carry greater default risk but may amplify equity returns. In the cross-section, leverage is a key input to the Quality factor composite and helps the ML model distinguish between cheap-for-good-reason (distressed) and genuinely undervalued stocks.
- **Profitability**: Pre-computed ROE percentage from SEC financials (`ROE_PCT / 100`), matching the production Quality factor composite. ROE captures how efficiently a company converts equity into profit — the academic "profitability premium" (Novy-Marx, 2013) shows high-ROE firms earn persistent excess returns. This feeds the Quality exposure in Phase 2's Fama-MacBeth regressions and serves as a standalone feature in Phase 3's XGBoost model.

In [13]:
fin = session.table(f"{DATABASE}.{MARKET_DATA}.FACT_SEC_FINANCIALS")

leverage_raw = (fin
    .join(dim_sec, fin["ISSUERID"] == dim_sec["ISSUERID"], "inner")
    .filter(F.col("COVERED_QTRS") == 0)
    .filter(F.col("DEBT_TO_EQUITY").is_not_null())
    .with_column("MONTH_DATE", F.date_trunc("MONTH", F.col("PERIOD_END_DATE")))
    .group_by(dim_sec["SECURITYID"], F.col("MONTH_DATE"))
    .agg(F.max("DEBT_TO_EQUITY").alias("LEVERAGE_RAW"))
    .select(
        F.col("SECURITYID").alias("LEV_SID")
        , F.col("MONTH_DATE").alias("LEV_MONTH")
        , F.col("LEVERAGE_RAW")
    )
)
print("Leverage features:")
print(leverage_raw.show())

prof_raw = (fin
    .join(dim_sec, fin["ISSUERID"] == dim_sec["ISSUERID"], "inner")
    .filter(F.col("COVERED_QTRS").isin([1, 4]))
    .with_column("MONTH_DATE", F.date_trunc("MONTH", F.col("PERIOD_END_DATE")))
    .group_by(dim_sec["SECURITYID"], F.col("MONTH_DATE"))
    .agg(F.max(F.coalesce(F.col("ROE_PCT"), F.lit(0)) / F.lit(100.0)).alias("PROFITABILITY_RAW"))
    .select(
        F.col("SECURITYID").alias("PROF_SID")
        , F.col("MONTH_DATE").alias("PROF_MONTH")
        , F.col("PROFITABILITY_RAW")
    )
)

print("Profitability features:")
prof_raw.show()

Leverage features:
-------------------------------------------------
|"LEV_SID"  |"LEV_MONTH"  |"LEVERAGE_RAW"       |
-------------------------------------------------
|140        |2025-12-01   |0.0                  |
|60         |2025-12-01   |0.21071274298056156  |
|500        |2024-12-01   |0.08015258161737618  |
|65         |2025-12-01   |0.7251777050660048   |
|101        |2024-12-01   |0.13159106019536643  |
|117        |2025-12-01   |0.6343862866055611   |
|226        |2025-12-01   |1.0984010923666476   |
|210        |2025-12-01   |2.4920789327404114   |
|225        |2023-12-01   |0.01899795554225827  |
|213        |2025-12-01   |0.3048495091984658   |
-------------------------------------------------

None
Profitability features:
------------------------------------------------------
|"PROF_SID"  |"PROF_MONTH"  |"PROFITABILITY_RAW"     |
------------------------------------------------------
|83          |2022-12-01    |0.16958516164954562     |
|354         |2025-12-01    |0.

#### Earnings Revision — quarter-over-quarter EPS change

Source: SEC financial statements (diluted EPS, quarterly filings). Compute lagged EPS via `analytics.compute_lag()`, then derive the percentage change. Guards against division by near-zero EPS. Earnings revisions capture the market's tendency to under-react to earnings surprises — the "post-earnings-announcement drift" (PEAD) is one of the oldest documented anomalies. A positive EPS revision signals improving fundamentals ahead of full price adjustment. This factor enters the Fama-MacBeth model as part of the fundamental factor set and gives XGBoost a signal orthogonal to price-based momentum.

In [12]:
eps_quarterly = (fin
    .join(dim_sec, fin["ISSUERID"] == dim_sec["ISSUERID"], "inner")
    .filter((F.col("COVERED_QTRS") == 1) & F.col("EPS_DILUTED").is_not_null())
    .with_column("MONTH_DATE", F.date_trunc("MONTH", F.col("PERIOD_END_DATE")))
    .group_by(dim_sec["SECURITYID"], F.col("MONTH_DATE"))
    .agg(F.max("EPS_DILUTED").alias("EPS_QUARTERLY"))
    .select(
        F.col("SECURITYID").alias("ER_SID")
        , F.col("MONTH_DATE").alias("ER_MONTH")
        , F.col("EPS_QUARTERLY")
    )
)

er_with_lag = eps_quarterly.analytics.compute_lag(
    cols=["EPS_QUARTERLY"], lags=[1], order_by=["ER_MONTH"], group_by=["ER_SID"]
    , col_formatter=lambda col, op, val: f"{col}_LAG_{val}"
)
er_revision = (er_with_lag
    .with_column("EARNINGS_REVISION_RAW",
        F.when(F.abs(F.col("EPS_QUARTERLY_LAG_1")) > 0.01,
            (F.col("EPS_QUARTERLY") - F.col("EPS_QUARTERLY_LAG_1")) / F.abs(F.col("EPS_QUARTERLY_LAG_1"))
        ).otherwise(F.lit(None))
    )
    .select("ER_SID", "ER_MONTH", "EARNINGS_REVISION_RAW")
)
er_revision.show()

---------------------------------------------------
|"ER_SID"  |"ER_MONTH"  |"EARNINGS_REVISION_RAW"  |
---------------------------------------------------
|487       |2021-06-01  |NULL                     |
|487       |2021-09-01  |-1.6578947368421053      |
|487       |2022-03-01  |-3.0533333333333332      |
|487       |2022-06-01  |-0.1809210526315789      |
|487       |2022-09-01  |0.25905292479108627      |
|487       |2023-03-01  |-0.2105263157894737      |
|487       |2023-06-01  |0.8726708074534161       |
|487       |2023-09-01  |-2.4146341463414633      |
|487       |2024-03-01  |-0.4714285714285716      |
|487       |2024-06-01  |0.8349514563106796       |
---------------------------------------------------



#### Dividend Yield — trailing 12-month DPS / close price

Source: Dividend payments + daily stock prices. Aggregate monthly dividends per share, compute a rolling 12-month trailing sum, then divide by the monthly close price. Dividend yield serves dual roles: as a value signal (high-yield stocks tend to be cheap relative to fundamentals) and as a defensive quality indicator (firms that sustain dividends typically have stable cash flows). The trailing 12-month window smooths seasonality in payout schedules. This factor feeds the Fundamental FeatureView, contributes to Phase 2's cross-sectional regressions, and gives Phase 3's XGBoost a distinct income-oriented signal separate from earnings-based value measures.

In [ ]:
divs = session.table(f"{DATABASE}.{MARKET_DATA}.FACT_DIVIDENDS")
div_trailing = (divs
    .with_column("MONTH_DATE", F.date_trunc("MONTH", F.col("EX_DATE")))
    .group_by("SECURITYID", "MONTH_DATE")
    .agg(F.sum("DIVIDEND_PER_SHARE").alias("MONTHLY_DPS"))
)

div_t12m = (div_trailing
    .with_column("TRAILING_12M_DPS",
        F.sum("MONTHLY_DPS").over(
            W.partition_by("SECURITYID")
            .order_by("MONTH_DATE")
            .rows_between(-11, W.CURRENT_ROW)
        )
    )
    .select(
        F.col("SECURITYID").alias("DIV_SID")
        , F.col("MONTH_DATE").alias("DIV_MONTH")
        , F.col("TRAILING_12M_DPS")
    )
)

div_yield = (div_t12m
    .join(price_features.select(
            F.col("SEC_ID").alias("MP_SID")
            , F.col("MONTH_DATE").alias("MP_MONTH")
            , F.col("CLOSE_PRICE")
        )
        , (div_t12m["DIV_SID"] == F.col("MP_SID"))
          & (div_t12m["DIV_MONTH"] == F.col("MP_MONTH"))
        , "inner"
    )
    .with_column("DIVIDEND_YIELD_RAW",
        F.col("TRAILING_12M_DPS") / F.nullif(F.col("CLOSE_PRICE"), F.lit(0))
    )
    .select("DIV_SID", "DIV_MONTH", "DIVIDEND_YIELD_RAW")
)
div_yield.show()

#### Join Fundamental Factors into Raw

Left-join four computed factors (leverage, profitability, earnings revision, dividend yield) into the `raw` DataFrame by security and month. Volatility and Beta are already present from the factor exposure pivot. Quarterly factors are **forward-filled** per security with a **6-month staleness guard** — values older than 6 months are nulled out, matching the production pipeline's join window. Non-dividend-paying stocks get `DIVIDEND_YIELD_RAW = 0`. Forward-filling is critical because SEC filings arrive quarterly while our factor model runs monthly — without it, 2 out of every 3 months would have NULL fundamentals. The staleness guard prevents outdated values from persisting indefinitely (e.g., if a company stops filing). The resulting DataFrame is the complete raw feature matrix that feeds into z-score normalisation and ultimately the Feature Store.

In [ ]:
raw_joined = (raw
    .join(leverage_raw, (raw["SEC_ID"] == leverage_raw["LEV_SID"]) & (raw["MONTH_DATE"] == leverage_raw["LEV_MONTH"]), "left")
    .join(prof_raw, (raw["SEC_ID"] == prof_raw["PROF_SID"]) & (raw["MONTH_DATE"] == prof_raw["PROF_MONTH"]), "left")
    .join(er_revision, (raw["SEC_ID"] == er_revision["ER_SID"]) & (raw["MONTH_DATE"] == er_revision["ER_MONTH"]), "left")
    .join(div_yield, (raw["SEC_ID"] == div_yield["DIV_SID"]) & (raw["MONTH_DATE"] == div_yield["DIV_MONTH"]), "left")
)

ffill_window = W.partition_by("SEC_ID").order_by("MONTH_DATE")
ffill_cols = ["LEVERAGE_RAW", "PROFITABILITY_RAW", "EARNINGS_REVISION_RAW", "DIVIDEND_YIELD_RAW"]
STALE_MONTHS = 6

raw_ffilled = raw_joined
for col_name in ffill_cols:
    raw_ffilled = raw_ffilled.with_column(f"_{col_name}_OBS",
        F.when(F.col(col_name).is_not_null(), F.col("MONTH_DATE"))
    )
    raw_ffilled = raw_ffilled.with_column(f"_{col_name}_OBS",
        F.coalesce(F.col(f"_{col_name}_OBS"),
            F.lag(F.col(f"_{col_name}_OBS"), ignore_nulls=True).over(ffill_window))
    )
    raw_ffilled = raw_ffilled.with_column(col_name,
        F.coalesce(F.col(col_name),
            F.lag(F.col(col_name), ignore_nulls=True).over(ffill_window))
    )
    raw_ffilled = raw_ffilled.with_column(col_name,
        F.when(F.datediff("month", F.col(f"_{col_name}_OBS"), F.col("MONTH_DATE")) <= STALE_MONTHS,
               F.col(col_name))
    )

raw_with_fundamentals = (raw_ffilled
    .with_column("DIVIDEND_YIELD_RAW", F.coalesce(F.col("DIVIDEND_YIELD_RAW"), F.lit(0)))
    .select("TICKER", "MONTH_DATE", "SEC_ID"
        , "MOMENTUM_1M", "MOMENTUM_3M", "MOMENTUM_12M"
        , "AVG_VOLUME", "MARKET_CAP_PROXY", "VOLUME_TREND"
        , "VALUE_EXPOSURE", "SIZE_EXPOSURE", "QUALITY_EXPOSURE", "GROWTH_EXPOSURE"
        , "VOLATILITY_EXPOSURE", "MARKET_EXPOSURE"
        , "LEVERAGE_RAW", "PROFITABILITY_RAW", "EARNINGS_REVISION_RAW", "DIVIDEND_YIELD_RAW"
    )
)

print(f"Raw features with fundamentals: {raw_with_fundamentals.count()} rows")
raw_with_fundamentals.select("VOLATILITY_EXPOSURE", "MARKET_EXPOSURE", "LEVERAGE_RAW", "PROFITABILITY_RAW", "EARNINGS_REVISION_RAW", "DIVIDEND_YIELD_RAW").describe().show()

#### Factor Quality Gate: Fundamental Coverage and Forward-Fill Health

SEC financial data arrives quarterly, but our factor model runs monthly. Forward-filling bridges this gap — but it can mask problems. If a factor has poor raw coverage, forward-filling propagates stale or missing values, injecting noise rather than signal.

**What to look for:**
- **Raw coverage** — What percentage of security-months have an actual filing value (before forward-fill)? Leverage and profitability should cover >60% of issuers per quarter.
- **Forward-fill effectiveness** — How many NULLs did the fill close? A large gap between raw and filled coverage indicates heavy reliance on stale data.
- **Distribution stability** — Boxplots over time should show stable medians and spreads. A sudden shift (e.g., leverage spiking) may be a real economic event or a data quality issue.

**Decision impact:** Factors with <30% coverage after forward-fill add more noise than signal — consider dropping them or replacing with a proxy. Factors where >50% of filled values are stale (>3 months old) should be treated with caution in the ML model.

In [ ]:
fund_cols = ["LEVERAGE_RAW", "PROFITABILITY_RAW", "EARNINGS_REVISION_RAW", "DIVIDEND_YIELD_RAW"]

before_counts = raw_joined.select(
    F.count("*").alias("TOTAL"),
    *[F.sum(F.when(F.col(c).is_not_null(), 1).otherwise(0)).alias(f"{c}_BEFORE") for c in fund_cols]
).to_pandas()
after_counts = raw_with_fundamentals.select(
    F.count("*").alias("TOTAL"),
    *[F.sum(F.when(F.col(c).is_not_null(), 1).otherwise(0)).alias(f"{c}_AFTER") for c in fund_cols]
).to_pandas()

total = int(before_counts["TOTAL"].iloc[0])

print(f"{'Factor':<25} {'Raw Coverage':>14} {'After FFill':>14} {'Filled by FFill':>16} {'Status':>10}")
print("-" * 85)
for c in fund_cols:
    raw_n = int(before_counts[f"{c}_BEFORE"].iloc[0])
    filled_n = int(after_counts[f"{c}_AFTER"].iloc[0])
    raw_pct = raw_n / total * 100
    filled_pct = filled_n / total * 100
    ffill_pct = (filled_n - raw_n) / total * 100
    status = "PASS" if filled_pct > 50 else "REVIEW" if filled_pct > 30 else "FAIL"
    label = c.replace("_RAW", "")
    print(f"{label:<25} {raw_pct:>12.1f}% {filled_pct:>12.1f}% {ffill_pct:>14.1f}% {status:>10}")

boxplot_sf = (raw_with_fundamentals
    .with_column("QUARTER", F.date_trunc("QUARTER", F.col("MONTH_DATE")))
    .group_by("QUARTER")
    .agg(*[agg_fn(F.col(c)).alias(f"{c}_{tag}")
           for c in fund_cols
           for agg_fn, tag in [
               (F.min, "MIN"), (F.median, "MED"), (F.max, "MAX"),
               (lambda col: F.call_builtin("PERCENTILE_CONT", 0.25, F.call_builtin("WITHIN GROUP", col)), "Q1"),
           ]]
    )
)

fund_pd = raw_with_fundamentals.select(
    F.date_trunc("QUARTER", F.col("MONTH_DATE")).alias("QUARTER"), *fund_cols
).to_pandas()
fund_pd["QUARTER"] = fund_pd["QUARTER"].astype(str).str[:10]

quarters = sorted(fund_pd["QUARTER"].unique())
sample_q = quarters[::max(1, len(quarters) // 8)]

fig = make_subplots(rows=1, cols=4,
    subplot_titles=[c.replace("_RAW", "") for c in fund_cols])
for i, c in enumerate(fund_cols):
    subset = fund_pd[fund_pd["QUARTER"].isin(sample_q)]
    for q in sample_q:
        q_vals = subset.loc[subset["QUARTER"] == q, c].dropna()
        fig.add_trace(go.Box(y=q_vals, name=q, marker_color="#3498db",
            opacity=0.6, showlegend=False), row=1, col=i+1)
    fig.update_xaxes(tickangle=45, tickfont_size=7, row=1, col=i+1)

fig.update_layout(height=400, template="plotly_white",
    title_text="Fundamental Factor Distributions Over Time (sampled quarters)", title_font_size=13)
fig.show()

### Cross-Sectional Z-Score Normalisation

A reusable `zscore()` function computes cross-sectional z-scores per month with ±3σ winsorisation (matching the production pipeline). Factor exposures from `FACT_FACTOR_EXPOSURES` (Value, Size, Quality, Growth, Volatility, Market) are already z-scored and winsorised by the data generation pipeline — these pass through with a ±3σ clamp only. Raw features (momentum, volume, leverage, profitability, earnings revision, dividend yield) are z-scored from scratch. Forward-filled fundamentals use a 6-month staleness guard matching the production join window.

**Why z-score?** Raw factor values live on incompatible scales — momentum is a percentage, leverage is a ratio, and market cap is in billions. Without normalisation, any regression or ML model would be dominated by the largest-magnitude features. Cross-sectional z-scoring (within each month) ensures every factor has zero mean and unit variance, making regression coefficients directly comparable in Phase 2 and giving XGBoost equal opportunity to learn from each feature in Phase 3. Winsorisation at ±3σ prevents extreme outliers (e.g., a stock with 50x leverage) from distorting the distribution.

In [ ]:
DECIMAL_TYPE = T.DecimalType(38, 10)

def zscore(df, cols, id_cols=("TICKER", "MONTH_DATE", "SEC_ID")):
    fixed = df
    for col_name, _ in cols:
        fixed = fixed.with_column(col_name, F.col(col_name).cast(DECIMAL_TYPE))

    monthly_stats = (fixed
        .group_by("MONTH_DATE")
        .agg(*[fn(col_name).alias(f"{col_name}_{stat}")
               for col_name, _ in cols
               for fn, stat in [(F.avg, "MEAN"), (F.stddev, "STD")]])
    )

    scored = fixed.join(monthly_stats, "MONTH_DATE", "inner")
    for col_name, alias in cols:
        scored = scored.with_column(alias,
            F.greatest(F.lit(-3), F.least(F.lit(3),
                F.when(F.col(f"{col_name}_STD") > 0,
                    (F.col(col_name) - F.col(f"{col_name}_MEAN")) / F.col(f"{col_name}_STD")
                ).otherwise(F.lit(0))
            )).cast(DECIMAL_TYPE)
        )

    return (scored
        .select(*list(id_cols), *[alias for _, alias in cols])
        .filter(F.col("TICKER").is_not_null())
    )

market_zscore_cols = [
    ("MOMENTUM_1M", "MOMENTUM_SCORE"),
    ("AVG_VOLUME", "LIQUIDITY_SCORE"),
    ("VOLUME_TREND", "VOLUME_TREND_SCORE"),
]

pre_zscored_market_renames = [
    ("VOLATILITY_EXPOSURE", "VOLATILITY_SCORE"),
    ("MARKET_EXPOSURE", "BETA_SCORE"),
]

fundamental_zscore_cols = [
    ("LEVERAGE_RAW", "LEVERAGE_SCORE"),
    ("PROFITABILITY_RAW", "PROFITABILITY_SCORE"),
    ("EARNINGS_REVISION_RAW", "EARNINGS_REVISION"),
    ("DIVIDEND_YIELD_RAW", "DIVIDEND_YIELD_SCORE"),
]

pre_zscored_fundamental_renames = [
    ("VALUE_EXPOSURE", "VALUE_SCORE"),
    ("QUALITY_EXPOSURE", "QUALITY_SCORE"),
    ("GROWTH_EXPOSURE", "GROWTH_SCORE"),
    ("SIZE_EXPOSURE", "SIZE_SCORE"),
]

market_extra_ids = ("TICKER", "MONTH_DATE", "SEC_ID", "VOLATILITY_EXPOSURE", "MARKET_EXPOSURE")
market_zscored = zscore(raw_with_fundamentals, market_zscore_cols, id_cols=market_extra_ids)
for src, dst in pre_zscored_market_renames:
    market_zscored = market_zscored.with_column(dst,
        F.greatest(F.lit(-3), F.least(F.lit(3), F.col(src).cast(DECIMAL_TYPE))))
market_factor_df = (market_zscored
    .select("TICKER", "MONTH_DATE", "SEC_ID",
            *[alias for _, alias in market_zscore_cols + pre_zscored_market_renames])
    .with_column("SECURITYID", F.col("SEC_ID")).drop("SEC_ID")
)

fund_extra_ids = ("TICKER", "MONTH_DATE", "SEC_ID", "VALUE_EXPOSURE", "QUALITY_EXPOSURE", "GROWTH_EXPOSURE", "SIZE_EXPOSURE")
fund_zscored = zscore(raw_with_fundamentals, fundamental_zscore_cols, id_cols=fund_extra_ids)
for src, dst in pre_zscored_fundamental_renames:
    fund_zscored = fund_zscored.with_column(dst,
        F.greatest(F.lit(-3), F.least(F.lit(3), F.col(src).cast(DECIMAL_TYPE))))
fundamental_factor_df = (fund_zscored
    .select("TICKER", "MONTH_DATE", "SEC_ID",
            *[alias for _, alias in fundamental_zscore_cols + pre_zscored_fundamental_renames])
    .with_column("SECURITYID", F.col("SEC_ID")).drop("SEC_ID")
)

factor_df = (market_factor_df
    .join(fundamental_factor_df, ["TICKER", "MONTH_DATE", "SECURITYID"], "inner")
)

all_market = len(market_zscore_cols) + len(pre_zscored_market_renames)
all_fund = len(fundamental_zscore_cols) + len(pre_zscored_fundamental_renames)
print(f"Market factors: {market_factor_df.count()} rows, {all_market} features ({len(market_zscore_cols)} z-scored + {len(pre_zscored_market_renames)} pre-scored from exposures)")
print(f"Fundamental factors: {fundamental_factor_df.count()} rows, {all_fund} features ({len(fundamental_zscore_cols)} z-scored + {len(pre_zscored_fundamental_renames)} pre-scored from exposures)")
factor_df.show()

#### Factor Quality Gate: Redundancy and Multicollinearity

If two factors are highly correlated, they carry overlapping information — including both adds complexity without improving predictive power, and makes Fama-MacBeth regression coefficients unstable (inflated standard errors). This check identifies redundant factor pairs before they enter the model.

**What to look for:**
- **Pairwise Spearman correlation** — Factor pairs with |rho| > 0.7 are candidates for merging or dropping one. Moderate correlation (0.3-0.7) is acceptable and common (e.g., value and quality often share some signal).
- **Condition number** — A summary measure of multicollinearity across all factors. Values > 30 indicate severe multicollinearity; > 10 warrants attention.

**Decision impact:** If two factors have |rho| > 0.7, keep the one with higher univariate IC (from the next check) and drop the other. If the condition number exceeds 30, consider PCA or dropping the most correlated factor. Moderate correlations (0.3-0.5) are normal in equity factor models and do not require action.

In [ ]:
factor_pd = factor_df.to_pandas()
score_cols = [c for c in factor_pd.columns if c.endswith("_SCORE")]

corr = factor_pd[score_cols].corr(method="spearman")

labels = [c.replace("_SCORE", "") for c in score_cols]
text_vals = [[f"{corr.values[r, c]:.2f}" for c in range(len(score_cols))] for r in range(len(score_cols))]

fig = go.Figure(data=go.Heatmap(
    z=corr.values, x=labels, y=labels,
    colorscale="RdBu_r", zmin=-1, zmax=1,
    text=text_vals, texttemplate="%{text}", textfont_size=9,
    colorbar=dict(title="ρ")))
fig.update_layout(height=550, width=650, template="plotly_white",
    title_text="Cross-Factor Spearman Correlation (13 Quant Factors)", title_font_size=12,
    xaxis=dict(tickangle=45, tickfont_size=8), yaxis=dict(tickfont_size=8))
fig.show()

cond_number = np.linalg.cond(factor_pd[score_cols].dropna().values)
high_pairs = []
for i in range(len(score_cols)):
    for j in range(i + 1, len(score_cols)):
        if abs(corr.values[i, j]) > 0.7:
            high_pairs.append((labels[i], labels[j], corr.values[i, j]))

print(f"\nCondition number: {cond_number:.1f} {'PASS' if cond_number < 30 else 'REVIEW' if cond_number < 100 else 'FAIL'}")
print(f"\nHighly correlated pairs (|rho| > 0.7):")
if high_pairs:
    for a, b, rho in high_pairs:
        print(f"  {a} <-> {b}: rho = {rho:+.3f}  REVIEW")
else:
    print("  None found  PASS")

#### Factor Quality Gate: Univariate Predictive Power (All 13 Quant Factors)

This is the single most important pre-model diagnostic. For each of the 13 z-scored quant factors, we compute the monthly Spearman rank correlation between the factor score and the 1-month forward return, then average across months. This "Information Coefficient" (IC) measures whether higher factor scores predict higher future returns — the core premise of factor investing.

**What to look for:**
- **Mean IC magnitude** — |IC| > 0.03 is a strong signal, 0.02-0.03 is moderate, < 0.01 is weak. Negative IC means the factor predicts returns in the opposite direction (still useful if stable).
- **IC t-statistic** — Is the mean IC statistically significant? |t| > 2.0 means 95% confidence that the factor has non-zero predictive power.
- **IC stability over time** — The monthly IC series should not be dominated by a few extreme months. A factor that predicts well in 3 months and fails in 57 is unreliable.

**Decision impact:** Factors with |mean IC| < 0.01 and |t| < 1.0 are unlikely to add standalone value. We still carry them forward because XGBoost may find non-linear interactions — but they are flagged for scrutiny in Phase 3's SHAP analysis. Factors with strong, stable IC are the candidates we expect to see validated as significant in Fama-MacBeth (Phase 2).

In [ ]:
price_fwd = (mp_agg
    .with_column("FWD_RET_1M",
        F.lead("CLOSE_PRICE", 1).over(W.partition_by("TICKER").order_by("MONTH_DATE"))
        / F.col("CLOSE_PRICE") - 1)
    .select("TICKER", "MONTH_DATE", "FWD_RET_1M")
)

ic_base = (factor_df
    .join(price_fwd, ["TICKER", "MONTH_DATE"], "left")
    .filter(F.col("FWD_RET_1M").is_not_null())
)

ic_ranked = ic_base
for c in score_cols:
    ic_ranked = ic_ranked.with_column(
        f"RANK_{c}", sf_rank().over(W.partition_by("MONTH_DATE").order_by(c)))
ic_ranked = (ic_ranked
    .with_column("RANK_FWD", sf_rank().over(W.partition_by("MONTH_DATE").order_by("FWD_RET_1M")))
    .with_column("N", F.count("*").over(W.partition_by("MONTH_DATE")))
    .filter(F.col("N") >= 10)
)

all_score_ics = (ic_ranked
    .group_by("MONTH_DATE")
    .agg(*[sf_corr(F.col(f"RANK_{c}"), F.col("RANK_FWD")).alias(f"IC_{c}") for c in score_cols])
    .sort("MONTH_DATE")
).to_pandas()

monthly_ics = {}
for col_name in score_cols:
    monthly_ics[col_name] = all_score_ics[f"IC_{col_name}"].dropna().tolist()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Mean Rank IC (green=strong, yellow=moderate, red=weak)",
                    "Rolling 12-Month IC (top 6 factors)"))

labels = [c.replace("_SCORE", "") for c in score_cols]
mean_ics = [np.mean(monthly_ics[c]) if monthly_ics[c] else 0 for c in score_cols]
ic_tstats = [np.mean(monthly_ics[c]) / (np.std(monthly_ics[c]) / np.sqrt(len(monthly_ics[c])))
             if monthly_ics[c] and np.std(monthly_ics[c]) > 0 else 0 for c in score_cols]

colors = ["#2ecc71" if abs(ic) > 0.02 else "#f39c12" if abs(ic) > 0.01 else "#e74c3c" for ic in mean_ics]
sigs = ["***" if abs(ts) > 2.58 else "**" if abs(ts) > 1.96 else "*" if abs(ts) > 1.28 else ""
        for ts in ic_tstats]
fig.add_trace(go.Bar(x=labels, y=mean_ics, marker_color=colors,
    text=[f"{ic:.3f}{s}" for ic, s in zip(mean_ics, sigs)],
    textposition="outside", textfont_size=8, showlegend=False), row=1, col=1)
fig.add_hline(y=0, line_width=0.5, row=1, col=1)
fig.add_hline(y=0.02, line_dash="dash", line_color="green", opacity=0.3, row=1, col=1)
fig.add_hline(y=-0.02, line_dash="dash", line_color="green", opacity=0.3, row=1, col=1)
fig.update_yaxes(title_text="Mean Spearman IC", row=1, col=1)
fig.update_xaxes(tickangle=45, tickfont_size=8, row=1, col=1)

for col_name in score_cols[:6]:
    if monthly_ics[col_name]:
        rolling = pd.Series(monthly_ics[col_name]).rolling(12, min_periods=3).mean()
        fig.add_trace(go.Scatter(x=list(range(len(rolling))), y=rolling,
            mode="lines", name=col_name.replace("_SCORE", ""), line=dict(width=1.2)), row=1, col=2)
fig.add_hline(y=0, line_dash="dash", line_color="black", opacity=0.3, row=1, col=2)
fig.update_yaxes(title_text="Rolling Mean IC", row=1, col=2)

fig.update_layout(height=400, template="plotly_white", legend=dict(font_size=8))
fig.show()

print(f"\n{'Factor':<22} {'Mean IC':>10} {'t-stat':>10} {'# Months':>10} {'Verdict':>10}")
print("-" * 65)
for i, col_name in enumerate(score_cols):
    n = len(monthly_ics[col_name])
    verdict = "STRONG" if abs(mean_ics[i]) > 0.02 else "MODERATE" if abs(mean_ics[i]) > 0.01 else "WEAK"
    print(f"{labels[i]:<22} {mean_ics[i]:>+10.4f} {ic_tstats[i]:>10.2f} {n:>10} {verdict:>10}")

### Phase 1.5: NLP Sentiment Factors from Earnings Transcripts

Traditional quant factors capture price-based and fundamental signals. Can unstructured text add marginal alpha? We use the Snowpark `ai_sentiment()` function with quant-relevant categories (guidance, margins, growth, risk) to extract multi-dimensional sentiment from earnings call transcripts (filtered to earnings calls, last 5 years).

**Numeric encoding**: `ai_sentiment()` returns categorical labels per category — we convert to: positive → +1, negative → −1, neutral → 0, mixed → 0, unknown → NULL. When multiple transcripts exist for the same security-month, we aggregate via `AVG` to produce a continuous score in [−1, +1]. These 5 sentiment dimensions (overall + 4 categories) join the factor model as additional features — XGBoost + SHAP will tell us whether they add predictive power after controlling for the 13 quantitative factors.

**No forward-fill**: Unlike fundamental factors (leverage, ROE) which represent persistent state and are forward-filled between filings, sentiment is a point-in-time event signal. A bullish earnings call from 6 months ago is not a reliable signal today. Months without a transcript have NULL sentiment — this is intentional. XGBoost handles missing values natively and can learn that "no recent signal" is itself informative. This also prevents stale sentiment from contaminating derived features like `SENTIMENT_MOMENTUM_INTERACTION`.

In [ ]:
from snowflake.snowpark.functions import ai_sentiment, array_to_string, ai_complete, ai_agg

transcripts_w_speaker = (session.table(f"{DATABASE}.CURATED.COMPANY_EVENT_TRANSCRIPTS_CORPUS")
    .filter(F.col("PUBLISH_DATE") >= F.dateadd("year", F.lit(-5), F.current_date()))
    .filter(F.col("EVENT_TYPE") == "Earnings Call")
)

sentiment_scored = (transcripts_w_speaker
                        .group_by(["SECURITYID", "TICKER", "PUBLISH_DATE"])
                        .agg(ai_agg(F.col("DOCUMENT_TEXT"), 
                                F.lit('Score the sentiments for  guidance, margins, growth, risk of this earnings call transcript on a -1.000-+1.000 scale using 3 decimals'
                                    '(-1.000=extremely negative, 0.000=neutral, +1.000=extremely positive). '
                                    'Each score must have a brief explanation of why the score. '
                                    'Return ONLY a JSON object with these keys: '
                                    'overall_score, overall_explain, guidance_score, guidance_explain, margins_score, margins_explain, growth_score, growth_explain, risk_score, risk_explain. '
                                    'Each value must be a float -1.000 - +1.000. '
                                    'Example: {"overall_score":0.543,"guidance_score":0.986,"margins_score":-0.765,"growth_score":0.987,"risk_score":-0.001}'
                                )
                            ).alias("LLM_RESPONSE")
                        )
                    )

sentiment_parsed = (
    sentiment_scored
    .with_column('RESP_JSON', F.parse_json(F.col('LLM_RESPONSE')))
    .with_column('SENT_OVERALL_SCORE', F.col('RESP_JSON')['SENTIMENT_OVERALL'].cast(T.FloatType()))
    .with_column('SENT_OVERALL_EXPL', F.col('RESP_JSON')['SENTIMENT_OVERALL_EXPLAIN'].cast(T.StringType()))
    .with_column('SENT_GUIDANCE_SCORE', F.col('RESP_JSON')['SENTIMENT_GUIDANCE'].cast(T.FloatType()))
    .with_column('SENT_GUIDANCE_EXPL', F.col('RESP_JSON')['SENTIMENT_GUIDANCE_EXPLAIN'].cast(T.StringType()))
    .with_column('SENT_MARGINS_SCORE', F.col('RESP_JSON')['SENTIMENT_MARGINS'].cast(T.FloatType()))
    .with_column('SENT_MARGINS_EXPL', F.col('RESP_JSON')['SENTIMENT_MARGINS_EXPLAIN'].cast(T.StringType()))
    .with_column('SENT_GROWTH_SCORE', F.col('RESP_JSON')['SENTIMENT_GROWTH'].cast(T.FloatType()))
    .with_column('SENT_GROWTH_EXPL', F.col('RESP_JSON')['SENTIMENT_GROWTH_EXPLAIN'].cast(T.StringType()))
    .with_column('SENT_RISK_SCORE', F.col('RESP_JSON')['SENTIMENT_RISK'].cast(T.FloatType()))
    .with_column('SENT_RISK_EXPL', F.col('RESP_JSON')['SENTIMENT_RISK_EXPLAIN'].cast(T.StringType()))
)

sentiment_parsed.limit(5).show()

In [ ]:
coverage = sentiment_parsed.count()
total = sentiment_parsed.select("SECURITYID", "MONTH_DATE").distinct().count()
print(f"Sentiment coverage: {coverage}/{total} security-months ({coverage/total*100:.1f}%)")

### Feature Store Initialisation

Connect to the Feature Store schema. Uses `CREATE_IF_NOT_EXIST` so re-runs are idempotent. The Feature Store is the backbone of this workflow — it manages versioned, entity-keyed feature tables backed by Dynamic Tables that auto-refresh on schedule. By centralising all factor data here, we ensure that Phase 2 (Fama-MacBeth), Phase 3 (XGBoost), and Phase 4 (portfolio construction) all consume the same consistent feature set, eliminating train/serve skew and making the research-to-production transition seamless.

In [ ]:
from snowflake.ml.feature_store import FeatureStore, Entity, FeatureView, CreationMode

fs = FeatureStore(
    session=session,
    database=DATABASE,
    name=ML_SCHEMA,
    default_warehouse="SAM_DEMO_EXECUTION_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

print(f"Feature Store: {DATABASE}.{ML_SCHEMA}")

### Register Entity

The entity represents the subject — a security identified by `SECURITYID`. All FeatureViews are keyed by this entity plus a timestamp (`MONTH_DATE`), enabling point-in-time correct joins when generating training datasets. This ensures that when we retrieve features for a given security at a given date, we get the values that were actually known at that time — preventing look-ahead bias in backtests.

In [ ]:
security_entity = Entity(
    name="SECURITY"
    , join_keys=["SECURITYID"]
    , desc="Individual equity security from SAM's investable universe (~100 tickers)."
)
with warnings.catch_warnings():
    warnings.simplefilter("ignore", UserWarning)
    try:
        fs.register_entity(security_entity)
        print(f"Entity registered: {security_entity.name}")
    except Exception:
        security_entity = fs.get_entity("SECURITY")
        print(f"Entity already exists, retrieved: {security_entity.name}")

### Register FeatureViews

Split factors into three domain-specific FeatureViews aligned with source data update cadences:
- **SECURITY_MARKET_FACTORS** (daily): momentum, volatility, beta, liquidity, volume trend
- **SECURITY_FUNDAMENTAL_FACTORS** (60 days): value, size, quality, growth, leverage, profitability, earnings revision, dividend yield
- **SECURITY_SENTIMENT_FACTORS** (weekly): 5 NLP sentiment dimensions via AI_SENTIMENT

Cast `MONTH_DATE` to `TIMESTAMP` (required by Feature Store) and register the factor scores as a versioned FeatureView backed by a managed Dynamic Table.

In [ ]:
version_name = "V01"

def get_latest_fv(fs, name):
    versions = fs.list_feature_views(feature_view_name=name)
    return sorted(versions, key=lambda fv: fv.version, reverse=True)[0] if versions else None

market_fv_df = market_factor_df.with_column("MONTH_DATE", F.col("MONTH_DATE").cast(T.TimestampType()))

market_fv = FeatureView(
    name="SECURITY_MARKET_FACTORS"
    , entities=[security_entity]
    , feature_df=market_fv_df
    , timestamp_col="MONTH_DATE"
    , refresh_freq="1 day"
    , warehouse="SAM_DEMO_EXECUTION_WH"
    , desc="""Daily market-derived factor scores (z-scored cross-sectionally).
Features:
- MOMENTUM_SCORE: 1-month price return
- VOLATILITY_SCORE: Monthly return std deviation (from factor exposures)
- BETA_SCORE: Monthly CAPM beta vs SPX (from factor exposures)
- LIQUIDITY_SCORE: Average daily trading volume
- VOLUME_TREND_SCORE: Month-over-month total volume change
Sources: Daily stock prices, daily returns, factor exposures
SLA: Daily refresh"""
)
registered_market_fv = fs.register_feature_view(market_fv, version=version_name, overwrite=True)
registered_market_fv.attach_feature_desc({
    "MOMENTUM_SCORE": "1-month price return z-score",
    "VOLATILITY_SCORE": "Monthly return stddev z-score (from factor exposures)",
    "BETA_SCORE": "Monthly CAPM beta vs SPX z-score (from factor exposures)",
    "LIQUIDITY_SCORE": "Average daily trading volume z-score",
    "VOLUME_TREND_SCORE": "Month-over-month total volume change z-score",
})
print(f"FeatureView registered: SECURITY_MARKET_FACTORS/{version_name}")

In [ ]:
fundamental_fv_df = fundamental_factor_df.with_column("MONTH_DATE", F.col("MONTH_DATE").cast(T.TimestampType()))

fundamental_fv = FeatureView(
    name="SECURITY_FUNDAMENTAL_FACTORS"
    , entities=[security_entity]
    , feature_df=fundamental_fv_df
    , timestamp_col="MONTH_DATE"
    , refresh_freq="60 days"
    , warehouse="SAM_DEMO_EXECUTION_WH"
    , desc="""Quarterly/monthly fundamental factor scores (z-scored cross-sectionally).
Features:
- VALUE_SCORE, SIZE_SCORE, QUALITY_SCORE, GROWTH_SCORE: Cybersyn factor exposures (monthly)
- LEVERAGE_SCORE: Debt-to-equity ratio (quarterly, SEC financial statements)
- PROFITABILITY_SCORE: ROE (quarterly, SEC financial statements)
- EARNINGS_REVISION: QoQ diluted EPS change (quarterly, SEC financial statements)
- DIVIDEND_YIELD_SCORE: Trailing 12M dividend yield (quarterly, dividend payments)
Sources: Factor exposures, SEC financial statements, dividend payments
SLA: Refreshes every 60 days"""
)
registered_fundamental_fv = fs.register_feature_view(fundamental_fv, version=version_name, overwrite=True)
registered_fundamental_fv.attach_feature_desc({
    "VALUE_SCORE": "Cybersyn Value factor exposure z-score",
    "SIZE_SCORE": "Cybersyn Size factor exposure z-score",
    "QUALITY_SCORE": "Cybersyn Quality factor exposure z-score",
    "GROWTH_SCORE": "Cybersyn Growth factor exposure z-score",
    "LEVERAGE_SCORE": "Debt-to-equity ratio z-score (quarterly)",
    "PROFITABILITY_SCORE": "ROE z-score (quarterly)",
    "EARNINGS_REVISION": "QoQ diluted EPS change z-score",
    "DIVIDEND_YIELD_SCORE": "Trailing 12M dividend yield z-score",
})
print(f"FeatureView registered: SECURITY_FUNDAMENTAL_FACTORS/{version_name}")

In [ ]:
sentiment_factor_df = (sentiment_df
    .select("SECURITYID", "MONTH_DATE",
        "SENTIMENT_OVERALL", "SENTIMENT_GUIDANCE", "SENTIMENT_MARGINS",
        "SENTIMENT_GROWTH", "SENTIMENT_RISK")
    .with_column("MONTH_DATE", F.col("MONTH_DATE").cast(T.TimestampType()))
)

sentiment_fv = FeatureView(
    name="SECURITY_SENTIMENT_FACTORS"
    , entities=[security_entity]
    , feature_df=sentiment_factor_df
    , timestamp_col="MONTH_DATE"
    , refresh_freq="7 days"
    , warehouse="SAM_DEMO_CORTEX_WH"
    , desc="""Event-driven NLP sentiment from earnings transcripts via AI_SENTIMENT.
Features:
- SENTIMENT_OVERALL: Overall transcript sentiment
- SENTIMENT_GUIDANCE: Forward guidance sentiment
- SENTIMENT_MARGINS: Margin outlook sentiment
- SENTIMENT_GROWTH: Growth narrative sentiment
- SENTIMENT_RISK: Risk language sentiment
Sources: Earnings call transcripts (via AI_SENTIMENT LLM function)
SLA: Weekly refresh (event-driven source, earnings are quarterly)"""
)
registered_sentiment_fv = fs.register_feature_view(sentiment_fv, version=version_name, overwrite=True)
print(f"FeatureView registered: SECURITY_SENTIMENT_FACTORS/{version_name}")

### Derived Features FeatureView

Interaction features derived from the 3 base FeatureViews (MIT — Model-Independent Transforms):
- **MOMENTUM_VALUE_RATIO**: Momentum / Value — momentum trap signal
- **QUALITY_GROWTH_INTERACTION**: Quality × Growth — GARP signal
- **SENTIMENT_MOMENTUM_INTERACTION**: NLP sentiment × momentum — confirmation signal
- **FACTOR_DISPERSION**: Max − Min of core factors — conviction indicator

In [ ]:
mkt = fs.read_feature_view(registered_market_fv)
fund = fs.read_feature_view(registered_fundamental_fv)
sent = fs.read_feature_view(registered_sentiment_fv)

base_factors = (mkt
    .join(fund, ["SECURITYID", "MONTH_DATE"], "inner")
    .join(sent, ["SECURITYID", "MONTH_DATE"], "left")
)

derived_df = (base_factors
    .with_column("MOMENTUM_VALUE_RATIO",
        F.when(F.col("VALUE_SCORE") != 0, F.col("MOMENTUM_SCORE") / F.col("VALUE_SCORE"))
         .otherwise(F.lit(None).cast(DECIMAL_TYPE))
    )
    .with_column("QUALITY_GROWTH_INTERACTION",
        (F.col("QUALITY_SCORE") * F.col("GROWTH_SCORE")).cast(DECIMAL_TYPE)
    )
    .with_column("SENTIMENT_MOMENTUM_INTERACTION",
        (F.col("SENTIMENT_OVERALL") * F.col("MOMENTUM_SCORE")).cast(DECIMAL_TYPE)
    )
    .with_column("FACTOR_DISPERSION",
        (F.greatest(F.col("MOMENTUM_SCORE"), F.col("VALUE_SCORE"), F.col("QUALITY_SCORE"))
        - F.least(F.col("MOMENTUM_SCORE"), F.col("VALUE_SCORE"), F.col("QUALITY_SCORE"))).cast(DECIMAL_TYPE)
    )
    .select("SECURITYID", "MONTH_DATE",
        "MOMENTUM_VALUE_RATIO", "QUALITY_GROWTH_INTERACTION",
        "SENTIMENT_MOMENTUM_INTERACTION", "FACTOR_DISPERSION")
)

derived_fv = FeatureView(
    name="SECURITY_DERIVED_FEATURES"
    , entities=[security_entity]
    , feature_df=derived_df
    , timestamp_col="MONTH_DATE"
    , refresh_freq="1 day"
    , warehouse="SAM_DEMO_EXECUTION_WH"
    , desc="""Derived factor interactions from 3 base FeatureViews.
Features:
- MOMENTUM_VALUE_RATIO: Momentum / Value -- momentum trap signal
- QUALITY_GROWTH_INTERACTION: Quality x Growth -- GARP signal
- SENTIMENT_MOMENTUM_INTERACTION: NLP sentiment x momentum -- confirmation signal
- FACTOR_DISPERSION: Max-Min of core factors -- conviction indicator
Sources: SECURITY_MARKET_FACTORS, SECURITY_FUNDAMENTAL_FACTORS, SECURITY_SENTIMENT_FACTORS FeatureViews"""
)

registered_derived_fv = fs.register_feature_view(derived_fv, version=version_name, overwrite=True)
print(f"FeatureView registered: SECURITY_DERIVED_FEATURES/{version_name}")

### Phase 1.8: Hidden Thematic Factor Construction

Can we find exposures **not captured** by the 7 standard factors (Value, Size, Quality, Growth, Momentum, Volatility, Market)? We construct 5 hidden thematic factors from SEC filings, earnings transcripts, and ESG data — the same alternative datasets already in our Snowflake account:

| Factor | Signal | Source |
|---|---|---|
| **AI_Exposure** | Revenue from AI/ML segments + NLP AI mention density | SEC segment revenue + earnings transcript NLP scores |
| **Reshoring_Benefit** | Domestic revenue concentration × sector boost | SEC segment geography + issuer GICS sector |
| **Rate_Convexity** | Leverage × short-term repricing exposure | SEC financial statements (debt structure) |
| **Climate_Transition** | Green transition exposure from sector + ESG environmental | Issuer GICS sector + ESG environmental scores |
| **Geopolitical_Risk** | Revenue in high-risk regions + NLP geopolitical risk score | SEC segment geography + earnings transcript NLP scores |

All factors are cross-sectionally z-scored with ±3σ winsorisation, matching the normalisation used for the 7 standard factors above.

In [ ]:
security_issuer = (
    session.table(f"{DATABASE}.{CURATED}.DIM_SECURITY").alias("s")
    .join(session.table(f"{DATABASE}.{CURATED}.DIM_ISSUER").alias("i"),
          F.col("s.IssuerID") == F.col("i.IssuerID"))
    .filter(F.col("s.AssetClass") == "Equity")
    .select(
        F.col("s.SecurityID").alias("SI_SECURITYID")
        , F.col("s.IssuerID").alias("SI_ISSUERID")
        , F.col("i.GICS_SECTOR").alias("SI_GICS_SECTOR")
    )
).cache_result()

segments = session.table(f"{DATABASE}.{MARKET_DATA}.FACT_SEC_SEGMENTS")
nlp_scores = session.table(f"{DATABASE}.{MARKET_DATA}.FACT_TRANSCRIPT_NLP_SCORES")
financials = session.table(f"{DATABASE}.{MARKET_DATA}.FACT_SEC_FINANCIALS")
esg_scores = session.table(f"{DATABASE}.{CURATED}.FACT_ESG_SCORES")

ai_seg = (segments
    .filter(F.col("BUSINESS_SEGMENT").is_not_null() & (F.col("SEGMENT_REVENUE") > 0))
    .with_column("FQ", F.quarter(F.col("PERIOD_END_DATE")))
    .group_by("ISSUERID", "FISCAL_YEAR", "FQ")
    .agg(
        (F.sum(F.when(F.col("AI_REVENUE_FLAG"), F.col("SEGMENT_REVENUE")).otherwise(F.lit(0)))
         / F.call_builtin("NULLIF", F.sum("SEGMENT_REVENUE"), F.lit(0))
        ).alias("AI_SEGMENT_SHARE")
    )
)

ai_ts = (nlp_scores
    .filter(F.col("AI_EXPOSURE_SCORE").is_not_null())
    .select(
        F.col("ISSUERID").alias("AT_ISSUERID")
        , F.col("FISCAL_YEAR").alias("AT_FY")
        , F.col("FISCAL_QUARTER").alias("AT_FQ")
        , (F.col("AI_EXPOSURE_SCORE") / F.lit(100.0)).alias("AI_TRANSCRIPT_SCORE")
    )
)

ai_raw = (security_issuer
    .join(ai_seg
        , (F.col("SI_ISSUERID") == ai_seg["ISSUERID"]), "left")
    .join(ai_ts
        , (F.col("SI_ISSUERID") == ai_ts["AT_ISSUERID"])
          & (F.coalesce(ai_seg["FISCAL_YEAR"], ai_ts["AT_FY"]) == ai_ts["AT_FY"])
          & (F.coalesce(ai_seg["FQ"], ai_ts["AT_FQ"]) == ai_ts["AT_FQ"])
        , "left")
    .with_column("FISCAL_YEAR_AI", F.coalesce(ai_seg["FISCAL_YEAR"], ai_ts["AT_FY"]))
    .with_column("RAW_SCORE",
        F.when(F.col("AI_SEGMENT_SHARE").is_not_null() & F.col("AI_TRANSCRIPT_SCORE").is_not_null(),
               F.col("AI_SEGMENT_SHARE") * 0.6 + F.col("AI_TRANSCRIPT_SCORE") * 0.4)
         .when(F.col("AI_SEGMENT_SHARE").is_not_null(), F.col("AI_SEGMENT_SHARE"))
         .when(F.col("AI_TRANSCRIPT_SCORE").is_not_null(), F.col("AI_TRANSCRIPT_SCORE"))
    )
    .filter(F.col("RAW_SCORE").is_not_null())
    .select(
        F.col("SI_SECURITYID").alias("SECURITYID")
        , F.col("FISCAL_YEAR_AI").alias("FISCAL_YEAR")
        , F.col("RAW_SCORE")
        , F.lit("AI_Exposure").alias("FACTOR_NAME")
    )
)

geo_nlp = (nlp_scores
    .filter(F.col("GEO_RISK_SCORE").is_not_null())
    .select(
        F.col("ISSUERID").alias("GN_ISSUERID")
        , F.col("FISCAL_YEAR").alias("GN_FY")
        , F.col("FISCAL_QUARTER").alias("GN_FQ")
        , (F.col("GEO_RISK_SCORE") / F.lit(100.0)).alias("GEO_RISK_NLP")
    )
)

HIGH_RISK = ".*(CHINA|HONG KONG|TAIWAN|RUSSIA|IRAN|MIDDLE EAST|ISRAEL).*"
MED_RISK = ".*(ASIA|LATIN|AFRICA|BRAZIL|INDIA|INDONESIA|MEXICO|TURKEY).*"
MED_EXCLUDE = ".*(CHINA|HONG KONG|TAIWAN|JAPAN|AUSTRALIA|SOUTH KOREA).*"

geo_sql = (segments
    .filter(F.col("GEOGRAPHY").is_not_null() & (F.col("SEGMENT_REVENUE") > 0))
    .with_column("FQ", F.quarter(F.col("PERIOD_END_DATE")))
    .with_column("HIGH_RISK_REV",
        F.when(F.call_builtin("RLIKE", F.upper(F.col("GEOGRAPHY")), F.lit(HIGH_RISK)), F.col("SEGMENT_REVENUE"))
         .otherwise(F.lit(0)))
    .with_column("MED_RISK_REV",
        F.when(F.call_builtin("RLIKE", F.upper(F.col("GEOGRAPHY")), F.lit(MED_RISK))
               & ~F.call_builtin("RLIKE", F.upper(F.col("GEOGRAPHY")), F.lit(MED_EXCLUDE)),
               F.col("SEGMENT_REVENUE"))
         .otherwise(F.lit(0)))
    .group_by("ISSUERID", "FISCAL_YEAR", "FQ")
    .agg(
        (F.sum("HIGH_RISK_REV") / F.call_builtin("NULLIF", F.sum("SEGMENT_REVENUE"), F.lit(0))).alias("HIGH_RISK_SHARE")
        , (F.sum("MED_RISK_REV") / F.call_builtin("NULLIF", F.sum("SEGMENT_REVENUE"), F.lit(0))).alias("MED_RISK_SHARE")
    )
)

geo_raw = (security_issuer
    .join(geo_nlp, F.col("SI_ISSUERID") == geo_nlp["GN_ISSUERID"], "left")
    .join(geo_sql
        , (F.col("SI_ISSUERID") == geo_sql["ISSUERID"])
          & (F.coalesce(geo_nlp["GN_FY"], geo_sql["FISCAL_YEAR"]) == geo_sql["FISCAL_YEAR"])
          & (F.coalesce(geo_nlp["GN_FQ"], geo_sql["FQ"]) == geo_sql["FQ"])
        , "left")
    .with_column("FISCAL_YEAR_GEO", F.coalesce(geo_nlp["GN_FY"], geo_sql["FISCAL_YEAR"]))
    .with_column("RAW_SCORE",
        F.coalesce(
            F.col("GEO_RISK_NLP"),
            F.col("HIGH_RISK_SHARE") * 1.0 + F.col("MED_RISK_SHARE") * 0.5
        ))
    .filter(F.col("RAW_SCORE").is_not_null())
    .select(
        F.col("SI_SECURITYID").alias("SECURITYID")
        , F.col("FISCAL_YEAR_GEO").alias("FISCAL_YEAR")
        , F.col("RAW_SCORE")
        , F.lit("Geopolitical_Risk").alias("FACTOR_NAME")
    )
)

DOMESTIC = ".*(UNITED STATES|DOMESTIC|NORTH AMERICA|U\\.S\\.).*"

reshoring_raw = (security_issuer
    .join(segments.filter(F.col("GEOGRAPHY").is_not_null() & (F.col("SEGMENT_REVENUE") > 0))
        , F.col("SI_ISSUERID") == segments["ISSUERID"])
    .with_column("DOM_REV",
        F.when(F.call_builtin("RLIKE", F.upper(F.col("GEOGRAPHY")), F.lit(DOMESTIC)), F.col("SEGMENT_REVENUE"))
         .otherwise(F.lit(0)))
    .with_column("FQ", F.quarter(F.col("PERIOD_END_DATE")))
    .group_by("SI_SECURITYID", "SI_GICS_SECTOR", "FISCAL_YEAR", "FQ")
    .agg(
        (F.sum("DOM_REV") / F.call_builtin("NULLIF", F.sum("SEGMENT_REVENUE"), F.lit(0))).alias("DOM_SHARE")
    )
    .with_column("RAW_SCORE",
        F.col("DOM_SHARE") * F.when(F.col("SI_GICS_SECTOR").isin("Industrials", "Materials"), F.lit(1.3)).otherwise(F.lit(1.0))
    )
    .filter(F.col("RAW_SCORE").is_not_null())
    .select(
        F.col("SI_SECURITYID").alias("SECURITYID")
        , F.col("FISCAL_YEAR")
        , F.col("RAW_SCORE")
        , F.lit("Reshoring_Benefit").alias("FACTOR_NAME")
    )
)

rate_latest = (financials
    .filter((F.col("FISCAL_PERIOD") != "FY") & F.col("TOTAL_EQUITY").is_not_null())
    .with_column("SHORT_TERM_RATIO",
        F.coalesce(F.col("CURRENT_LIABILITIES"), F.lit(0))
        / F.call_builtin("NULLIF", F.col("TOTAL_LIABILITIES"), F.lit(0))
    )
    .with_column("RN",
        F.row_number().over(W.partition_by("ISSUERID", "FISCAL_YEAR").order_by(F.col("PERIOD_END_DATE").desc())))
    .filter(F.col("RN") == 1)
)

rate_raw = (security_issuer
    .join(rate_latest, F.col("SI_ISSUERID") == rate_latest["ISSUERID"])
    .with_column("RAW_SCORE",
        F.coalesce(F.col("DEBT_TO_EQUITY"), F.lit(0)) * F.coalesce(F.col("SHORT_TERM_RATIO"), F.lit(0))
    )
    .filter(F.col("RAW_SCORE").is_not_null())
    .select(
        F.col("SI_SECURITYID").alias("SECURITYID")
        , rate_latest["FISCAL_YEAR"]
        , F.col("RAW_SCORE")
        , F.lit("Rate_Convexity").alias("FACTOR_NAME")
    )
)

sector_score_expr = (
    F.when(F.col("SI_GICS_SECTOR") == "Energy", F.lit(-0.8))
     .when(F.col("SI_GICS_SECTOR") == "Utilities", F.lit(-0.4))
     .when(F.col("SI_GICS_SECTOR") == "Materials", F.lit(-0.3))
     .when(F.col("SI_GICS_SECTOR") == "Industrials", F.lit(-0.1))
     .when(F.col("SI_GICS_SECTOR") == "Information Technology", F.lit(0.3))
     .when(F.col("SI_GICS_SECTOR") == "Healthcare", F.lit(0.1))
     .when(F.col("SI_GICS_SECTOR") == "Financials", F.lit(0.0))
     .when(F.col("SI_GICS_SECTOR") == "Consumer Staples", F.lit(-0.1))
     .when(F.col("SI_GICS_SECTOR") == "Consumer Discretionary", F.lit(0.0))
     .when(F.col("SI_GICS_SECTOR") == "Communication Services", F.lit(0.1))
     .when(F.col("SI_GICS_SECTOR") == "Real Estate", F.lit(-0.2))
     .otherwise(F.lit(0.0))
)

esg_env = (esg_scores
    .filter(F.col("SCORE_TYPE") == "Environmental")
    .group_by("SECURITYID")
    .agg(F.avg("SCORE_VALUE").alias("ENV_SCORE"))
)

climate_raw = (security_issuer
    .join(esg_env, F.col("SI_SECURITYID") == esg_env["SECURITYID"], "left")
    .with_column("RAW_SCORE",
        sector_score_expr * 0.6
        + F.coalesce(F.col("ENV_SCORE") / F.lit(100.0) - F.lit(0.5), F.lit(0)) * 0.8
    )
    .filter(F.col("RAW_SCORE").is_not_null())
    .select(
        F.col("SI_SECURITYID").alias("SECURITYID")
        , F.lit(None).cast(T.IntegerType()).alias("FISCAL_YEAR")
        , F.col("RAW_SCORE")
        , F.lit("Climate_Transition").alias("FACTOR_NAME")
    )
)

all_hidden = (
    ai_raw
    .union_all_by_name(geo_raw)
    .union_all_by_name(reshoring_raw)
    .union_all_by_name(rate_raw)
    .union_all_by_name(climate_raw)
)

hidden_stats = (all_hidden
    .group_by("FACTOR_NAME", "FISCAL_YEAR")
    .agg(F.avg("RAW_SCORE").alias("MEAN_SCORE"), F.stddev("RAW_SCORE").alias("STD_SCORE"))
    .filter(F.col("STD_SCORE") > 0)
)

hidden_zscored = (all_hidden.alias("af")
    .join(hidden_stats.alias("fs"),
          (F.col("af.FACTOR_NAME") == F.col("fs.FACTOR_NAME"))
          & (F.coalesce(F.col("af.FISCAL_YEAR"), F.lit(2025)) == F.coalesce(F.col("fs.FISCAL_YEAR"), F.lit(2025))))
    .with_column("Z_SCORE",
        F.greatest(F.lit(-3.0),
            F.least(F.lit(3.0),
                (F.col("af.RAW_SCORE") - F.col("fs.MEAN_SCORE")) / F.col("fs.STD_SCORE")
            )
        )
    )
    .select(
        F.col("af.SECURITYID")
        , F.col("af.FISCAL_YEAR")
        , F.col("af.FACTOR_NAME")
        , F.col("Z_SCORE")
    )
)

coverage = hidden_zscored.group_by("FACTOR_NAME").agg(
    F.count_distinct("SECURITYID").alias("N_SECURITIES")
    , F.count("*").alias("N_ROWS")
    , F.avg("Z_SCORE").alias("MEAN_Z")
    , F.stddev("Z_SCORE").alias("STD_Z")
)
print("Hidden factor coverage:")
coverage.show()

In [ ]:
HIDDEN_FACTORS = ["AI_Exposure", "Reshoring_Benefit", "Rate_Convexity", "Climate_Transition", "Geopolitical_Risk"]

hidden_wide = (hidden_zscored
    .pivot(F.col("FACTOR_NAME"), HIDDEN_FACTORS)
    .agg(F.max(F.col("Z_SCORE")))
    .select(
        F.col("SECURITYID").alias("H_SID")
        , F.col("FISCAL_YEAR").alias("H_FY")
        , F.col("'AI_Exposure'").alias("AI_EXPOSURE_SCORE")
        , F.col("'Reshoring_Benefit'").alias("RESHORING_SCORE")
        , F.col("'Rate_Convexity'").alias("RATE_CONVEXITY_SCORE")
        , F.col("'Climate_Transition'").alias("CLIMATE_SCORE")
        , F.col("'Geopolitical_Risk'").alias("GEOPOLITICAL_SCORE")
    )
)

hidden_factor_df = (factor_df
    .join(hidden_wide,
          (factor_df["SECURITYID"] == hidden_wide["H_SID"])
          & (F.year(factor_df["MONTH_DATE"]) == F.coalesce(hidden_wide["H_FY"], F.year(factor_df["MONTH_DATE"])))
        , "left")
    .select(
        factor_df["SECURITYID"]
        , factor_df["TICKER"]
        , factor_df["MONTH_DATE"]
        , F.coalesce(F.col("AI_EXPOSURE_SCORE"), F.lit(0.0)).alias("AI_EXPOSURE_SCORE")
        , F.coalesce(F.col("RESHORING_SCORE"), F.lit(0.0)).alias("RESHORING_SCORE")
        , F.coalesce(F.col("RATE_CONVEXITY_SCORE"), F.lit(0.0)).alias("RATE_CONVEXITY_SCORE")
        , F.coalesce(F.col("CLIMATE_SCORE"), F.lit(0.0)).alias("CLIMATE_SCORE")
        , F.coalesce(F.col("GEOPOLITICAL_SCORE"), F.lit(0.0)).alias("GEOPOLITICAL_SCORE")
    )
)

total_secs = factor_df.select("SECURITYID").distinct().count()
hidden_non_null = hidden_wide.count()
print(f"Hidden factors: {hidden_non_null} security-years with scores out of {total_secs} total securities")
hidden_factor_df.show(5)

### Hidden Factor Validation

Before including hidden factors in our model, we validate three properties:

1. **Coverage** — Do enough securities have non-null scores? (threshold: >50% of universe per month)
2. **Cross-sectional dispersion** — Is there enough spread to differentiate stocks? (std > 0.3 after z-scoring)
3. **Univariate IC** — Does each factor individually predict forward returns? (|rank IC| > 0.02 is promising)

We also check **factor correlation** against traditional factors — if a hidden factor is highly correlated with an existing one (|ρ| > 0.5), it may not add new information.

In [ ]:
hidden_pd = hidden_factor_df.to_pandas()
hidden_cols = ["AI_EXPOSURE_SCORE", "RESHORING_SCORE", "RATE_CONVEXITY_SCORE", "CLIMATE_SCORE", "GEOPOLITICAL_SCORE"]

fig = make_subplots(rows=1, cols=5, shared_yaxes=True,
    subplot_titles=["" for _ in hidden_cols])
for i, col_name in enumerate(hidden_cols):
    vals = hidden_pd[col_name].dropna()
    cov_pct = (vals != 0).sum() / len(vals) * 100
    fig.add_trace(go.Histogram(x=vals, nbinsx=30, opacity=0.7,
        name=col_name.replace("_SCORE", ""), showlegend=False), row=1, col=i+1)
    fig.add_vline(x=0, line_dash="dash", line_color="red", opacity=0.5, row=1, col=i+1)
    fig.update_xaxes(title_text=f"{col_name.replace('_SCORE','')}<br>cov={cov_pct:.0f}% std={vals.std():.2f}",
        title_font_size=9, row=1, col=i+1)
fig.update_layout(height=350, template="plotly_white",
    title_text="Hidden Factor Distributions (z-scored)", title_font_size=13)
fig.show()

print("\nCoverage & Dispersion Summary:")
print(f"{'Factor':<25} {'Non-zero%':>10} {'Std':>8} {'Skew':>8} {'Kurt':>8} {'Status':>10}")
print("-" * 75)
for col_name in hidden_cols:
    vals = hidden_pd[col_name]
    nz_pct = (vals != 0).sum() / len(vals) * 100
    status = "PASS" if nz_pct > 50 and vals.std() > 0.3 else "REVIEW"
    print(f"{col_name.replace('_SCORE',''):<25} {nz_pct:>9.1f}% {vals.std():>8.3f} {vals.skew():>8.3f} {vals.kurtosis():>8.3f} {status:>10}")

In [ ]:
trad_cols = ["MOMENTUM_SCORE", "VALUE_SCORE", "QUALITY_SCORE", "GROWTH_SCORE",
             "SIZE_SCORE", "VOLATILITY_SCORE"]

merged_sf = (hidden_factor_df
    .join(fwd_ret_df.select("SECURITYID", "TICKER", "MONTH_DATE", "FORWARD_RETURN_21D"),
          ["SECURITYID", "TICKER", "MONTH_DATE"], "inner")
    .join(factor_df.select("SECURITYID", "MONTH_DATE", *trad_cols),
          ["SECURITYID", "MONTH_DATE"], "left")
)

hidden_ranked = (merged_sf
    .filter(F.col("FORWARD_RETURN_21D").is_not_null())
)
for c in hidden_cols:
    hidden_ranked = hidden_ranked.with_column(
        f"RANK_{c}", sf_rank().over(W.partition_by("MONTH_DATE").order_by(c)))
hidden_ranked = (hidden_ranked
    .with_column("RANK_FWD", sf_rank().over(W.partition_by("MONTH_DATE").order_by("FORWARD_RETURN_21D")))
    .with_column("N", F.count("*").over(W.partition_by("MONTH_DATE")))
    .filter(F.col("N") >= 10)
)

all_hidden_ics = (hidden_ranked
    .group_by("MONTH_DATE")
    .agg(*[sf_corr(F.col(f"RANK_{c}"), F.col("RANK_FWD")).alias(f"IC_{c}") for c in hidden_cols])
    .sort("MONTH_DATE")
).to_pandas()

monthly_ic = {}
for col_name in hidden_cols:
    monthly_ic[col_name] = all_hidden_ics[f"IC_{col_name}"].dropna().tolist()

all_factor_cols = trad_cols + hidden_cols
corr_ranked = merged_sf
for c in all_factor_cols:
    corr_ranked = corr_ranked.with_column(f"GR_{c}", sf_rank().over(W.order_by(c)))
corr_aggs = []
for hc in hidden_cols:
    for tc in trad_cols:
        corr_aggs.append(sf_corr(F.col(f"GR_{hc}"), F.col(f"GR_{tc}")).alias(f"RHO_{hc}__{tc}"))
corr_result = corr_ranked.select(*corr_aggs).to_pandas().iloc[0]

corr_pairs = []
for hc in hidden_cols:
    for tc in trad_cols:
        rho = corr_result[f"RHO_{hc}__{tc}"]
        corr_pairs.append({"HIDDEN": hc, "TRAD": tc, "RHO": float(rho) if rho is not None else 0.0})

corr_df = pd.DataFrame(corr_pairs)
hidden_vs_trad = corr_df.pivot(index="HIDDEN", columns="TRAD", values="RHO").reindex(index=hidden_cols, columns=trad_cols)

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Univariate Rank IC vs 21-day Forward Return",
                    "Hidden vs Traditional Factor Correlation"),
    specs=[[{"type": "xy"}, {"type": "heatmap"}]])

factor_labels = [c.replace("_SCORE", "") for c in hidden_cols]
mean_ics = [np.mean(monthly_ic[c]) if monthly_ic[c] else 0 for c in hidden_cols]
ic_tstats = [np.mean(monthly_ic[c]) / (np.std(monthly_ic[c]) / np.sqrt(len(monthly_ic[c])))
             if monthly_ic[c] and np.std(monthly_ic[c]) > 0 else 0 for c in hidden_cols]

colors = ["#2ecc71" if abs(ic) > 0.02 else "#e74c3c" for ic in mean_ics]
sigs = ["***" if abs(ts) > 2.58 else "**" if abs(ts) > 1.96 else "*" if abs(ts) > 1.28 else ""
        for ts in ic_tstats]
fig.add_trace(go.Bar(x=factor_labels, y=mean_ics, marker_color=colors,
    text=[f"{ic:.4f}{s}" for ic, s in zip(mean_ics, sigs)],
    textposition="outside", textfont_size=9, showlegend=False), row=1, col=1)
fig.add_hline(y=0, line_width=0.5, row=1, col=1)
fig.add_hline(y=0.02, line_dash="dash", line_color="green", opacity=0.3, row=1, col=1)
fig.add_hline(y=-0.02, line_dash="dash", line_color="green", opacity=0.3, row=1, col=1)
fig.update_yaxes(title_text="Mean Spearman IC", row=1, col=1)

trad_labels = [c.replace("_SCORE", "") for c in trad_cols]
text_vals = [[f"{hidden_vs_trad.values[r, c]:.2f}" for c in range(len(trad_cols))] for r in range(len(hidden_cols))]
fig.add_trace(go.Heatmap(z=hidden_vs_trad.values, x=trad_labels, y=factor_labels,
    colorscale="RdBu_r", zmin=-0.5, zmax=0.5,
    text=text_vals, texttemplate="%{text}", textfont_size=9,
    colorbar=dict(title="ρ", x=1.02)), row=1, col=2)
fig.update_xaxes(tickangle=45, tickfont_size=8, row=1, col=2)

fig.update_layout(height=450, template="plotly_white")
fig.show()

print(f"\n{'Factor':<25} {'Mean IC':>10} {'t-stat':>10} {'Max |ρ| vs trad':>18} {'Verdict':>10}")
print("-" * 80)
for i, col_name in enumerate(hidden_cols):
    max_corr = hidden_vs_trad.iloc[i].abs().max()
    max_corr_name = trad_cols[hidden_vs_trad.iloc[i].abs().argmax()].replace("_SCORE", "")
    verdict = "INCLUDE" if (abs(mean_ics[i]) > 0.01 or max_corr < 0.3) else "EXCLUDE"
    print(f"{factor_labels[i]:<25} {mean_ics[i]:>+10.4f} {ic_tstats[i]:>10.2f} {max_corr:>8.2f} ({max_corr_name:<8}) {verdict:>10}")

In [ ]:
hidden_fv_df = (hidden_factor_df
    .select("SECURITYID", F.col("MONTH_DATE").cast(T.TimestampType()).alias("MONTH_DATE"),
            "AI_EXPOSURE_SCORE", "RESHORING_SCORE", "RATE_CONVEXITY_SCORE",
            "CLIMATE_SCORE", "GEOPOLITICAL_SCORE")
)

hidden_fv = FeatureView(
    name="SECURITY_HIDDEN_FACTORS"
    , entities=[security_entity]
    , feature_df=hidden_fv_df
    , timestamp_col="MONTH_DATE"
    , refresh_freq="60 days"
    , warehouse="SAM_DEMO_EXECUTION_WH"
    , desc="""5 hidden thematic factor z-scores from SEC filings, earnings transcripts, and ESG data.
Features:
- AI_EXPOSURE_SCORE: AI/ML revenue share + transcript NLP AI density
- RESHORING_SCORE: Domestic revenue concentration with sector boost
- RATE_CONVEXITY_SCORE: Leverage x short-term repricing ratio
- CLIMATE_SCORE: GICS sector carbon mapping + ESG environmental score
- GEOPOLITICAL_SCORE: High-risk geography revenue + NLP geo risk
Sources: SEC segment revenue, earnings transcript NLP scores, SEC financial statements, ESG scores
SLA: 60-day refresh (quarterly source cadence)"""
)

registered_hidden_fv = fs.register_feature_view(hidden_fv, version=version_name, overwrite=True)
registered_hidden_fv.attach_feature_desc({
    "AI_EXPOSURE_SCORE": "AI/ML revenue share + transcript NLP density z-score",
    "RESHORING_SCORE": "Domestic revenue concentration z-score (sector-boosted)",
    "RATE_CONVEXITY_SCORE": "Leverage x short-term repricing ratio z-score",
    "CLIMATE_SCORE": "Green transition exposure from GICS sector + ESG env z-score",
    "GEOPOLITICAL_SCORE": "High-risk geography revenue + NLP geo risk z-score",
})
print(f"FeatureView registered: SECURITY_HIDDEN_FACTORS/{version_name}")

### Forward Return Target FeatureView

Separate target variable into its own FeatureView to prevent train/serve skew and enable consistent forward return computation across Fama-MacBeth and XGBoost pipelines. The forward 21-day return (approximately 1 month of trading days) is the dependent variable we are trying to predict. Isolating it in a dedicated FeatureView ensures the target is never accidentally included as a feature during training, and that the same return calculation is used consistently across Phase 2 regressions and Phase 3 ML models. The 21-day horizon balances signal strength (long enough to capture factor-driven moves) against turnover costs (short enough for a monthly rebalance strategy).

In [ ]:
fwd_dim_sec = session.table(f"{DATABASE}.{CURATED}.DIM_SECURITY")
fwd_sec_ret = session.table(f"{DATABASE}.{CURATED}.V_SECURITY_RETURNS")

fwd_ret_df = (fwd_sec_ret
    .join(fwd_dim_sec, fwd_sec_ret["SECURITYID"] == fwd_dim_sec["SECURITYID"], "inner")
    .select(
        fwd_dim_sec["SECURITYID"].alias("FWD_SID"),
        fwd_dim_sec["TICKER"].alias("FWD_TICKER"),
        fwd_sec_ret["PRICE_DATE"],
        fwd_sec_ret["DAILY_RETURN_PCT"],
        F.lead(fwd_sec_ret["DAILY_RETURN_PCT"], 21)
            .over(Window.partition_by(fwd_dim_sec["TICKER"]).order_by(fwd_sec_ret["PRICE_DATE"]))
            .alias("FORWARD_RETURN")
    )
    .with_column("MONTH_DATE", F.date_trunc("MONTH", F.col("PRICE_DATE")).cast(T.TimestampType()))
    .group_by("FWD_SID", "FWD_TICKER", "MONTH_DATE")
    .agg(F.avg("FORWARD_RETURN").alias("FORWARD_RETURN_21D"))
    .select(
        F.col("FWD_SID").alias("SECURITYID")
        , F.col("FWD_TICKER").alias("TICKER")
        , F.col("MONTH_DATE")
        , F.col("FORWARD_RETURN_21D")
    )
)

returns_fv = FeatureView(
    name="SECURITY_FORWARD_RETURNS"
    , entities=[security_entity]
    , feature_df=fwd_ret_df
    , timestamp_col="MONTH_DATE"
    , refresh_freq="1 day"
    , warehouse="SAM_DEMO_EXECUTION_WH"
    , desc="21-day forward return (avg of daily lead-21). Used as label/target for factor models."
)

registered_returns_fv = fs.register_feature_view(returns_fv, version=version_name, overwrite=True)
print(f"FeatureView registered: SECURITY_FORWARD_RETURNS/{version_name}")

### Dynamic Table Health Check

Monitor FeatureView Dynamic Table refresh status and staleness. Each FeatureView is backed by a managed Dynamic Table that auto-refreshes according to its cadence (daily, weekly, or 60 days). Before proceeding to model training, we verify that all tables have refreshed successfully and that data is not stale — a failed or lagging refresh would mean training on outdated features, which silently degrades model quality. This check also surfaces scheduling or permission issues early, before they propagate into incorrect factor returns or portfolio weights.

In [ ]:
dt_health = session.sql(f"""
    SELECT NAME, SCHEDULING_STATE, DATA_TIMESTAMP,
           TIMESTAMPDIFF('minute', DATA_TIMESTAMP, CURRENT_TIMESTAMP()) AS LAG_MINUTES
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLES())
    WHERE SCHEMA_NAME = '{ML_SCHEMA}'
    ORDER BY NAME
""")
dt_health.show()

stale_count = dt_health.filter(F.col("LAG_MINUTES") > 1440).count()
print(f"Staleness check: {stale_count} DTs exceed 24h lag ({'PASS' if stale_count == 0 else 'WARNING'})")

## Phase 2: Cross-Sectional Factor Model (Fama-MacBeth)

Phase 2 asks: **are these factors economically meaningful?** We register a Python UDTF via `session.udtf.register()` that runs Fama-MacBeth cross-sectional regressions — for each month, regress forward stock returns on the eleven factor scores (6 traditional + 5 hidden thematic) across the full universe. The time-series of regression coefficients are the factor return premia; their t-statistics tell us which factors are statistically significant. This econometric validation establishes which factors are worth feeding into Phase 3's ML model.

In [ ]:
from typing import Iterable, Tuple
from datetime import date

class FamaMacBethHandler:
    def __init__(self):
        self._data = []

    def process(self, month_date, ticker, forward_return, mom, val, qual, gro, siz, vol, ai, resh, rate, clim, geo):
        self._data.append([month_date, ticker, forward_return, mom, val, qual, gro, siz, vol, ai, resh, rate, clim, geo])

    def end_partition(self) -> Iterable[Tuple[date, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float, float]]:
        import numpy as np
        import pandas as pd

        FACTORS = ['mom', 'val', 'qual', 'gro', 'siz', 'vol', 'ai', 'resh', 'rate', 'clim', 'geo']
        df = pd.DataFrame(self._data, columns=['month_date', 'ticker', 'y'] + FACTORS)

        for md, group in df.groupby('month_date'):
            y = group['y'].values
            X = group[FACTORS].values
            X = np.column_stack([np.ones(len(X)), X])

            mask = ~np.isnan(y) & ~np.any(np.isnan(X), axis=1)
            if mask.sum() < 10:
                continue
            y, X = y[mask], X[mask]

            try:
                beta = np.linalg.lstsq(X, y, rcond=None)[0]
                y_hat = X @ beta
                ss_res = np.sum((y - y_hat) ** 2)
                ss_tot = np.sum((y - y.mean()) ** 2)
                r2 = 1 - ss_res / max(ss_tot, 1e-10)
                residuals = y - y_hat
                mse = np.sum(residuals ** 2) / max(len(y) - X.shape[1], 1)
                var_beta = mse * np.linalg.inv(X.T @ X).diagonal()
                se = np.sqrt(np.abs(var_beta))
                tstats = beta / np.where(se > 0, se, 1e-10)

                yield (md,
                       *[float(beta[i+1]) for i in range(len(FACTORS))],
                       *[float(tstats[i+1]) for i in range(len(FACTORS))],
                       float(r2))
            except Exception:
                continue

session.sql(f"CREATE STAGE IF NOT EXISTS {DATABASE}.{ML_SCHEMA}.UDTF_STAGE").collect()

FACTOR_NAMES = ["MOMENTUM", "VALUE", "QUALITY", "GROWTH", "SIZE", "VOLATILITY",
                "AI_EXPOSURE", "RESHORING", "RATE_CONVEXITY", "CLIMATE", "GEOPOLITICAL"]

fama_macbeth_udtf = session.udtf.register(
    FamaMacBethHandler
    , output_schema=T.StructType(
        [T.StructField("MONTH_DATE", T.DateType())]
        + [T.StructField(f"{f}_RETURN", T.FloatType()) for f in FACTOR_NAMES]
        + [T.StructField(f"{f}_TSTAT", T.FloatType()) for f in FACTOR_NAMES]
        + [T.StructField("R_SQUARED", T.FloatType())]
    )
    , input_types=[T.DateType(), T.StringType()] + [T.FloatType()] * 12
    , name=f"{DATABASE}.{ML_SCHEMA}.FAMA_MACBETH_REGRESSION"
    , is_permanent=True
    , replace=True
    , packages=["numpy", "pandas"]
    , stage_location=f"@{DATABASE}.{ML_SCHEMA}.UDTF_STAGE"
)
print(f"UDTF registered: {DATABASE}.{ML_SCHEMA}.FAMA_MACBETH_REGRESSION")

### Run Fama-MacBeth Cross-Sectional Regressions

We join factor scores with forward 21-day returns, then invoke the `FAMA_MACBETH_REGRESSION` UDTF via `join_table_function()`. The UDTF runs one OLS regression per month — regressing forward returns on the eleven factor scores (6 traditional + 5 hidden thematic) — producing monthly factor return coefficients, t-statistics, and R² values. This is the econometric backbone: a factor's time-series average coefficient is its return premium (basis points per unit of exposure per month), and a t-statistic above ~2.0 means the premium is statistically distinguishable from zero. Only factors that pass this test are economically validated — the results directly inform which features we expect to matter most in Phase 3's ML model.

In [ ]:
dim_sec_fm = session.table(f"{DATABASE}.{CURATED}.DIM_SECURITY").select(
    F.col("TICKER").alias("DIM_TICKER")
    , F.col("SECURITYID").alias("DIM_SID")
)
sec_ret = session.table(f"{DATABASE}.{CURATED}.V_SECURITY_RETURNS")

scored = (factor_df
    .join(dim_sec_fm, factor_df["TICKER"] == dim_sec_fm["DIM_TICKER"], "left")
    .join(sec_ret
        , (dim_sec_fm["DIM_SID"] == sec_ret["SECURITYID"]) & (sec_ret["PRICE_DATE"] == factor_df["MONTH_DATE"])
        , "left")
    .join(hidden_factor_df.alias("hf")
        , (factor_df["SECURITYID"] == F.col("hf.SECURITYID"))
          & (factor_df["MONTH_DATE"] == F.col("hf.MONTH_DATE"))
        , "left")
    .select(
        F.lit(1).alias("PARTITION_COL")
        , factor_df["TICKER"].alias("INPUT_TICKER")
        , factor_df["MONTH_DATE"].alias("INPUT_MONTH_DATE")
        , factor_df["MOMENTUM_SCORE"].cast(T.FloatType()).alias("MOMENTUM_SCORE")
        , factor_df["VALUE_SCORE"].cast(T.FloatType()).alias("VALUE_SCORE")
        , factor_df["QUALITY_SCORE"].cast(T.FloatType()).alias("QUALITY_SCORE")
        , factor_df["GROWTH_SCORE"].cast(T.FloatType()).alias("GROWTH_SCORE")
        , factor_df["SIZE_SCORE"].cast(T.FloatType()).alias("SIZE_SCORE")
        , factor_df["VOLATILITY_SCORE"].cast(T.FloatType()).alias("VOLATILITY_SCORE")
        , F.coalesce(F.col("hf.AI_EXPOSURE_SCORE"), F.lit(0.0)).cast(T.FloatType()).alias("AI_EXPOSURE_SCORE")
        , F.coalesce(F.col("hf.RESHORING_SCORE"), F.lit(0.0)).cast(T.FloatType()).alias("RESHORING_SCORE")
        , F.coalesce(F.col("hf.RATE_CONVEXITY_SCORE"), F.lit(0.0)).cast(T.FloatType()).alias("RATE_CONVEXITY_SCORE")
        , F.coalesce(F.col("hf.CLIMATE_SCORE"), F.lit(0.0)).cast(T.FloatType()).alias("CLIMATE_SCORE")
        , F.coalesce(F.col("hf.GEOPOLITICAL_SCORE"), F.lit(0.0)).cast(T.FloatType()).alias("GEOPOLITICAL_SCORE")
        , F.lead(sec_ret["DAILY_RETURN_PCT"], 21)
            .over(W.partition_by(factor_df["TICKER"]).order_by(factor_df["MONTH_DATE"]))
            .alias("FORWARD_RETURN")
    )
)

factor_returns_raw = (scored
    .join_table_function(
        fama_macbeth_udtf(
            F.col("INPUT_MONTH_DATE"), F.col("INPUT_TICKER"), F.col("FORWARD_RETURN")
            , F.col("MOMENTUM_SCORE"), F.col("VALUE_SCORE"), F.col("QUALITY_SCORE")
            , F.col("GROWTH_SCORE"), F.col("SIZE_SCORE"), F.col("VOLATILITY_SCORE")
            , F.col("AI_EXPOSURE_SCORE"), F.col("RESHORING_SCORE"), F.col("RATE_CONVEXITY_SCORE")
            , F.col("CLIMATE_SCORE"), F.col("GEOPOLITICAL_SCORE")
        ).over(partition_by=F.col("PARTITION_COL"))
    )
)

factor_returns_sf = factor_returns_raw.select(
    F.col("MONTH_DATE")
    , *[F.col(f"{f}_RETURN") for f in FACTOR_NAMES]
    , *[F.col(f"{f}_TSTAT") for f in FACTOR_NAMES]
    , F.col("R_SQUARED")
).sort("MONTH_DATE")

factor_returns_df = factor_returns_sf.to_pandas()
factor_returns_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.FACT_FACTOR_RETURNS")
print(f"Factor returns: {len(factor_returns_df)} months")


### Factor Return Analysis

Cumulative factor returns reveal which style factors have delivered persistent premia over the sample period, while annualised Sharpe ratios quantify risk-adjusted performance. Factors with high Sharpe ratios and significant t-statistics (from the regression above) are validated as economically meaningful — these are the factors Phase 3 will use to build a return prediction model.

In [ ]:
factor_returns_df = factor_returns_sf.to_pandas()

fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Cumulative Factor Returns (solid=traditional, dashed=hidden)",
                    "Factor Sharpe Ratios (annualised, blue=traditional, orange=hidden)"))
trad_return_cols = [f"{f}_RETURN" for f in FACTOR_NAMES[:6]]
hidden_return_cols = [f"{f}_RETURN" for f in FACTOR_NAMES[6:]]

for f in trad_return_cols:
    cumret = (1 + factor_returns_df[f].fillna(0)).cumprod()
    fig.add_trace(go.Scatter(x=factor_returns_df["MONTH_DATE"], y=cumret,
        mode="lines", name=f.replace("_RETURN", ""), line=dict(width=1.5)), row=1, col=1)
for f in hidden_return_cols:
    cumret = (1 + factor_returns_df[f].fillna(0)).cumprod()
    fig.add_trace(go.Scatter(x=factor_returns_df["MONTH_DATE"], y=cumret,
        mode="lines", name=f.replace("_RETURN", ""), line=dict(width=1.5, dash="dash")), row=1, col=1)

all_return_cols = trad_return_cols + hidden_return_cols
sharpe = {}
for f in all_return_cols:
    ret = factor_returns_df[f].dropna()
    sharpe[f.replace("_RETURN", "")] = ret.mean() / max(ret.std(), 1e-10) * np.sqrt(12)
colors = ["#3498db"] * 6 + ["#e67e22"] * 5
fig.add_trace(go.Bar(x=list(sharpe.keys()), y=list(sharpe.values()),
    marker_color=colors, showlegend=False), row=1, col=2)
fig.update_xaxes(tickangle=45, tickfont_size=8, row=1, col=2)

fig.update_layout(height=450, template="plotly_white", legend=dict(font_size=7))
fig.show()

#### Model Diagnostics: Fama-MacBeth Regression Health

The Fama-MacBeth regressions tell us whether factors earn significant premia, but the regression itself may be poorly specified. These diagnostics check the model's assumptions and stability over time.

**What to look for:**
- **R-squared over time** — How much of cross-sectional return variance do the 11 factors explain each month? Typical values for equity factor models are 5-15%. Consistently below 3% suggests the factors have weak joint explanatory power.
- **R-squared stability** — Large spikes (>30%) in individual months often indicate overfitting to that cross-section or an anomalous market event. Many months near zero suggest the model is barely better than random.
- **Rolling factor premium stability** — A 12-month rolling average of factor return premia reveals whether premia are persistent or concentrated in specific periods. A factor that was strong historically but dead in recent periods may not be forward-looking.

**Decision impact:** If average R-squared < 3%, the linear factor model has weak explanatory power — but ML (Phase 3) may still find signal through non-linear interactions. If R-squared is highly unstable, consider regime-conditional models. Factors whose rolling premium has decayed to zero in recent periods warrant lower confidence.

In [ ]:
fig = make_subplots(rows=1, cols=2,
    subplot_titles=("Monthly Cross-Sectional R-squared", "Rolling 12-Month Factor Return Premium"))

mean_r2 = factor_returns_df["R_SQUARED"].mean() * 100
fig.add_trace(go.Scatter(x=factor_returns_df["MONTH_DATE"], y=factor_returns_df["R_SQUARED"] * 100,
    mode="lines+markers", marker=dict(size=3), name="R²", showlegend=False), row=1, col=1)
fig.add_hline(y=mean_r2, line_dash="dash", line_color="red", opacity=0.7,
    annotation_text=f"Mean = {mean_r2:.1f}%", row=1, col=1)
fig.add_hline(y=3, line_dash="dot", line_color="orange", opacity=0.5,
    annotation_text="3% floor", row=1, col=1)
fig.update_yaxes(title_text="R-squared (%)", row=1, col=1)

return_cols = [f"{f}_RETURN" for f in FACTOR_NAMES]
rolling_premia = pd.DataFrame({"MONTH_DATE": factor_returns_df["MONTH_DATE"]})
for col in return_cols:
    rolling_premia[col] = factor_returns_df[col].rolling(12, min_periods=3).mean()

hidden_set = {f"{f}_RETURN" for f in FACTOR_NAMES[6:]}
for col in return_cols:
    fig.add_trace(go.Scatter(x=rolling_premia["MONTH_DATE"], y=rolling_premia[col],
        mode="lines", name=col.replace("_RETURN", ""),
        line=dict(width=1, dash="dash" if col in hidden_set else "solid")), row=1, col=2)
fig.add_hline(y=0, line_width=0.5, row=1, col=2)
fig.update_yaxes(title_text="Rolling Mean Return", row=1, col=2)

fig.update_layout(height=400, template="plotly_white", legend=dict(font_size=7))
fig.show()

r2_verdicts = factor_returns_sf.select(
    F.avg("R_SQUARED").alias("AVG_R2"),
    F.sum(F.when(F.col("R_SQUARED") < 0.03, 1).otherwise(0)).alias("LOW_R2_MONTHS"),
    F.count("*").alias("TOTAL_MONTHS")
).to_pandas().iloc[0]

avg_r2 = float(r2_verdicts["AVG_R2"]) * 100
low_r2_months = int(r2_verdicts["LOW_R2_MONTHS"])
total_months = int(r2_verdicts["TOTAL_MONTHS"])

print(f"{'Metric':<40} {'Value':>10} {'Status':>10}")
print("-" * 65)
print(f"{'Avg R-squared':<40} {avg_r2:>9.1f}% {'PASS' if avg_r2 > 5 else 'REVIEW' if avg_r2 > 3 else 'WEAK':>10}")
print(f"{'Months with R-sq < 3%':<40} {low_r2_months:>10} {'REVIEW' if low_r2_months > total_months * 0.5 else 'PASS':>10}")
print(f"{'Total months':<40} {total_months:>10}")

## Phase 3: ML Factor Discovery — XGBoost + SHAP

Phase 2 confirmed which factors earn statistically significant premia. Phase 3 now asks: **how do we best combine them to predict individual stock returns?** XGBoost now has 27 features (13 quant + 5 NLP sentiment + 5 hidden thematic + 4 derived interactions). Linear regressions capture the average factor premium, but markets aren't linear — momentum may matter more for small-caps, or quality may dominate during high-volatility regimes. XGBoost captures these non-linear interactions, and SHAP explains which factors drive predictions and under what conditions. The 3-way SHAP comparison answers: does alternative data (NLP + hidden thematic) add alpha after controlling for known factors?

In [ ]:
from snowflake.ml.experiment import ExperimentTracking
from snowflake.ml.experiment.callback.xgboost import SnowflakeXgboostCallback
import xgboost as xgb
from sklearn.model_selection import TimeSeriesSplit
import shap

experiment = ExperimentTracking(
    session=session
    , database_name=DATABASE
    , schema_name=ML_SCHEMA
)
experiment.set_experiment("factor_discovery")

spine_df = market_factor_df.select("SECURITYID", "MONTH_DATE").distinct()

dataset = fs.generate_dataset(
    name=f"{DATABASE}.{ML_SCHEMA}.FACTOR_TRAINING_DS"
    , spine_df=spine_df
    , features=[registered_market_fv, registered_fundamental_fv, registered_sentiment_fv,
                registered_derived_fv, registered_hidden_fv, registered_returns_fv]
    , spine_timestamp_col="MONTH_DATE"
    , version="V01"
    , desc="Factor discovery training dataset from Feature Store"
)

raw_training = dataset.read.to_pandas()

training_data = (raw_training
    .merge(
        factor_returns_df[["MONTH_DATE"]].assign(HAS_RETURN=True),
        on="MONTH_DATE",
        how="inner"
    )
    .rename(columns={"FORWARD_RETURN_21D": "FORWARD_RETURN"})
)

feature_cols = ["MOMENTUM_SCORE", "VALUE_SCORE", "QUALITY_SCORE", "GROWTH_SCORE",
                "SIZE_SCORE", "VOLATILITY_SCORE", "PROFITABILITY_SCORE", "LEVERAGE_SCORE",
                "EARNINGS_REVISION", "DIVIDEND_YIELD_SCORE", "BETA_SCORE", "LIQUIDITY_SCORE",
                "VOLUME_TREND_SCORE",
                "SENTIMENT_OVERALL", "SENTIMENT_GUIDANCE", "SENTIMENT_MARGINS",
                "SENTIMENT_GROWTH", "SENTIMENT_RISK",
                "MOMENTUM_VALUE_RATIO", "QUALITY_GROWTH_INTERACTION",
                "SENTIMENT_MOMENTUM_INTERACTION", "FACTOR_DISPERSION",
                "AI_EXPOSURE_SCORE", "RESHORING_SCORE", "RATE_CONVEXITY_SCORE",
                "CLIMATE_SCORE", "GEOPOLITICAL_SCORE"]

X = training_data[feature_cols].fillna(0)
y = training_data["FORWARD_RETURN"].fillna(0)

leakage_count = (training_data["MONTH_DATE"] > pd.Timestamp.now()).sum()
print(f"Data leakage check: {leakage_count} future rows ({'PASS' if leakage_count == 0 else 'FAIL'})")
print(f"ML training set: {len(X)} samples, {X.shape[1]} features")

In [ ]:
xgb_callback = SnowflakeXgboostCallback(experiment)

params = {
    "objective": "reg:squarederror",
    "max_depth": 5,
    "learning_rate": 0.05,
    "n_estimators": 150,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "random_state": 42,
}
for k, v in params.items():
    experiment.log_param(k, v)

tscv = TimeSeriesSplit(n_splits=3)
ic_scores = []

for fold, (train_idx, val_idx) in enumerate(tscv.split(X)):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = xgb.XGBRegressor(**params, callbacks=[xgb_callback])
    model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

    y_pred = model.predict(X_val)
    ic = np.corrcoef(y_val, y_pred)[0, 1]
    ic_scores.append(ic)
    print(f"Fold {fold+1}: IC={ic:.4f}")

mean_ic = np.mean(ic_scores)
experiment.log_metric("mean_ic", mean_ic)

final_model = xgb.XGBRegressor(**params)
final_model.fit(X, y, verbose=False)
print(f"\nMean IC: {mean_ic:.4f}")

explainer = shap.TreeExplainer(final_model)
shap_values = explainer.shap_values(X)

fig, ax = plt.subplots(1, 1, figsize=(10, 8))
shap.summary_plot(shap_values, X, feature_names=feature_cols, show=False)
plt.title("SHAP Beeswarm — Factor Importance for Return Prediction")
plt.tight_layout()
plt.show()

#### Model Diagnostics: XGBoost Learning Quality

A positive aggregate IC does not guarantee a well-behaved model. These diagnostics check whether XGBoost is learning meaningful patterns or memorising noise — critical before using predictions for portfolio construction.

**What to look for:**
- **Actual vs predicted scatter** — Should show a positive linear relationship, even if noisy. If the scatter is a random cloud with no trend, the model has no practical predictive power despite a nominally positive IC.
- **Residual distribution** — Residuals (actual - predicted) should be roughly symmetric around zero with no systematic patterns. A skewed residual distribution means the model systematically over- or under-predicts for certain stocks.
- **Per-fold IC consistency** — IC should be similar across all cross-validation folds. If one fold has IC = 0.15 and another has IC = -0.02, the signal is regime-dependent and less trustworthy for live trading.
- **Feature importance concentration** — If >50% of SHAP importance is concentrated in 2-3 features, the model is fragile. A more distributed importance profile is more robust.

**Decision impact:** If the actual-vs-predicted scatter shows no relationship, do not use this model for portfolio construction — revisit feature engineering. If per-fold IC varies widely, consider regime-aware training or ensemble approaches. High residual skewness suggests the model struggles with outlier stocks (potential for tail risk in the portfolio).

In [ ]:
y_pred_all = final_model.predict(X)
residuals = y.values - y_pred_all

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    f"Actual vs Predicted (IC={np.corrcoef(y.values, y_pred_all)[0,1]:.4f})",
    f"Residual Distribution (skew={pd.Series(residuals).skew():.2f}, kurt={pd.Series(residuals).kurtosis():.2f})",
    "Per-Fold IC Consistency",
    "Top 10 Features by Mean |SHAP| (cumulative %)"])

overall_ic = np.corrcoef(y.values, y_pred_all)[0, 1]
z = np.polyfit(y_pred_all, y.values, 1)
fig.add_trace(go.Scatter(x=y_pred_all, y=y.values, mode="markers",
    marker=dict(size=2, opacity=0.15), name="Obs", showlegend=False), row=1, col=1)
sorted_pred = np.sort(y_pred_all)
fig.add_trace(go.Scatter(x=sorted_pred, y=np.polyval(z, sorted_pred),
    mode="lines", line=dict(color="red", width=2), name="Fit", showlegend=False), row=1, col=1)
fig.update_xaxes(title_text="Predicted Return", row=1, col=1)
fig.update_yaxes(title_text="Actual Return", row=1, col=1)

fig.add_trace(go.Histogram(x=residuals, nbinsx=50, opacity=0.7,
    name="Residuals", showlegend=False), row=1, col=2)
fig.add_vline(x=0, line_dash="dash", line_color="red", row=1, col=2)
fig.update_xaxes(title_text="Residual (actual - predicted)", row=1, col=2)

fold_labels = [f"Fold {i+1}" for i in range(len(ic_scores))]
fold_colors = ["#2ecc71" if ic > 0 else "#e74c3c" for ic in ic_scores]
fig.add_trace(go.Bar(x=fold_labels, y=ic_scores, marker_color=fold_colors,
    showlegend=False), row=2, col=1)
fig.add_hline(y=mean_ic, line_dash="dash", line_color="blue",
    annotation_text=f"Mean IC={mean_ic:.4f}", row=2, col=1)
fig.update_yaxes(title_text="Information Coefficient", row=2, col=1)

mean_abs_shap = np.mean(np.abs(shap_values), axis=0)
sorted_idx = np.argsort(mean_abs_shap)[::-1]
top_n = 10
top_features = [feature_cols[i] for i in sorted_idx[:top_n]]
top_importance = mean_abs_shap[sorted_idx[:top_n]]
cumul_pct = np.cumsum(top_importance) / mean_abs_shap.sum() * 100

fig.add_trace(go.Bar(y=top_features[::-1], x=top_importance[::-1], orientation="h",
    marker_color="#3498db", text=[f"{c:.0f}%" for c in cumul_pct[::-1]],
    textposition="outside", textfont_size=8, showlegend=False), row=2, col=2)
fig.update_xaxes(title_text="Mean |SHAP value|", row=2, col=2)

fig.update_layout(height=700, template="plotly_white",
    title_text="XGBoost Model Quality Diagnostics", title_font_size=14)
fig.show()

ic_std = np.std(ic_scores)
top3_pct = cumul_pct[2] if len(cumul_pct) >= 3 else 100
resid_skew = abs(pd.Series(residuals).skew())

print(f"\n{'Metric':<35} {'Value':>12} {'Status':>10}")
print("-" * 60)
print(f"{'Overall IC (in-sample)':<35} {overall_ic:>12.4f} {'PASS' if overall_ic > 0.05 else 'REVIEW':>10}")
print(f"{'Mean CV IC':<35} {mean_ic:>12.4f} {'PASS' if mean_ic > 0.02 else 'REVIEW':>10}")
print(f"{'IC std across folds':<35} {ic_std:>12.4f} {'PASS' if ic_std < 0.05 else 'REVIEW':>10}")
print(f"{'Residual |skewness|':<35} {resid_skew:>12.2f} {'PASS' if resid_skew < 1 else 'REVIEW':>10}")
print(f"{'Top-3 feature concentration':<35} {top3_pct:>11.0f}% {'REVIEW' if top3_pct > 60 else 'PASS':>10}")

### Does Alternative Data Add Alpha?

The key question for a quant: does alternative data add marginal predictive power beyond traditional factors? We train three XGBoost models — (1) quant-only, (2) quant + NLP sentiment, (3) full model with hidden thematic factors — and compare Information Coefficients. The 3-way SHAP comparison shows whether hidden factors (AI exposure, reshoring, climate transition) rank among the top drivers.

In [ ]:
HIDDEN_COLS = ["AI_EXPOSURE_SCORE", "RESHORING_SCORE", "RATE_CONVEXITY_SCORE", "CLIMATE_SCORE", "GEOPOLITICAL_SCORE"]

feature_cols_quant_only = [c for c in feature_cols
                           if not c.startswith("SENTIMENT_") and c not in HIDDEN_COLS]
feature_cols_quant_nlp = [c for c in feature_cols if c not in HIDDEN_COLS]

X_quant = training_data[feature_cols_quant_only].fillna(0)
X_quant_nlp = training_data[feature_cols_quant_nlp].fillna(0)

tscv_compare = TimeSeriesSplit(n_splits=3)
ic_quant_only, ic_quant_nlp = [], []
for train_idx, val_idx in tscv_compare.split(X_quant):
    m1 = xgb.XGBRegressor(**params)
    m1.fit(X_quant.iloc[train_idx], y.iloc[train_idx], verbose=False)
    ic_quant_only.append(np.corrcoef(y.iloc[val_idx], m1.predict(X_quant.iloc[val_idx]))[0, 1])

    m2 = xgb.XGBRegressor(**params)
    m2.fit(X_quant_nlp.iloc[train_idx], y.iloc[train_idx], verbose=False)
    ic_quant_nlp.append(np.corrcoef(y.iloc[val_idx], m2.predict(X_quant_nlp.iloc[val_idx]))[0, 1])

model_quant = xgb.XGBRegressor(**params)
model_quant.fit(X_quant, y, verbose=False)
model_quant_nlp = xgb.XGBRegressor(**params)
model_quant_nlp.fit(X_quant_nlp, y, verbose=False)

ic_uplift_nlp = np.mean(ic_quant_nlp) - np.mean(ic_quant_only)
ic_uplift_hidden = mean_ic - np.mean(ic_quant_nlp)

fig, axes = plt.subplots(1, 3, figsize=(26, 8))

plt.sca(axes[0])
shap_quant = shap.TreeExplainer(model_quant).shap_values(X_quant)
shap.summary_plot(shap_quant, X_quant, feature_names=feature_cols_quant_only, show=False, max_display=13)
axes[0].set_title(f"Quant Only (IC={np.mean(ic_quant_only):.4f})", fontsize=11)

plt.sca(axes[1])
shap_qn = shap.TreeExplainer(model_quant_nlp).shap_values(X_quant_nlp)
shap.summary_plot(shap_qn, X_quant_nlp, feature_names=feature_cols_quant_nlp, show=False, max_display=18)
axes[1].set_title(f"Quant + NLP (IC={np.mean(ic_quant_nlp):.4f})", fontsize=11)

plt.sca(axes[2])
shap.summary_plot(shap_values, X, feature_names=feature_cols, show=False, max_display=27)
axes[2].set_title(f"Full Model incl. Hidden (IC={mean_ic:.4f})", fontsize=11)

plt.suptitle(f"IC Uplift: NLP={ic_uplift_nlp:+.4f}  |  Hidden={ic_uplift_hidden:+.4f}  |  Total={mean_ic - np.mean(ic_quant_only):+.4f}",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

experiment.log_metric("ic_quant_only", float(np.mean(ic_quant_only)))
experiment.log_metric("ic_quant_nlp", float(np.mean(ic_quant_nlp)))
experiment.log_metric("ic_uplift_nlp", float(ic_uplift_nlp))
experiment.log_metric("ic_uplift_hidden", float(ic_uplift_hidden))
print(f"IC Quant only:       {np.mean(ic_quant_only):.4f}")
print(f"IC Quant + NLP:      {np.mean(ic_quant_nlp):.4f}")
print(f"IC Full (+ Hidden):  {mean_ic:.4f}")
print(f"IC uplift NLP:       {ic_uplift_nlp:+.4f}")
print(f"IC uplift Hidden:    {ic_uplift_hidden:+.4f}")

alt_data_cols = [c for c in feature_cols if c.startswith("SENTIMENT_") or c in HIDDEN_COLS]
alt_shap_rank = sorted(
    [(f, np.mean(np.abs(shap_values[:, i]))) for i, f in enumerate(feature_cols) if f in alt_data_cols],
    key=lambda x: -x[1]
)
print(f"\nAlternative data SHAP ranking (NLP + Hidden):")
for name, importance in alt_shap_rank:
    print(f"  {name}: mean|SHAP|={importance:.6f}")

## Phase 3 (cont): Model Registration

Register the trained XGBoost model in the Snowflake Model Registry with versioned metadata, IC (Information Coefficient), and turnover metrics. Registration creates a persistent, governed artifact that can be called for inference in SQL or Python without re-training. The IC metric measures rank correlation between predicted and realised returns (higher is better), while turnover measures how much the portfolio changes month-to-month (lower is cheaper to trade). Together these metrics enable model comparison across versions and inform the go/no-go decision for production deployment in Phase 4.

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(session=session, database_name=DATABASE, schema_name=ML_SCHEMA)

sample_input = session.create_dataframe(X.head(10))
version_name = "V01"

y_pred_all = final_model.predict(X)
turnover = np.abs(np.diff(y_pred_all)).mean()

model_version = registry.log_model(
    model=final_model,
    model_name="FACTOR_RETURN_XGBOOST",
    version_name=version_name,
    sample_input_data=sample_input,
    target_platforms=["WAREHOUSE"],
    metrics={
        "mean_ic": float(mean_ic),
        "turnover": float(turnover),
        "n_samples": len(X),
    },
    comment="XGBoost factor return prediction: 27 features (13 quant + 5 NLP + 5 hidden thematic + 4 derived) -> forward 21-day return"
)

print(f"Model logged: FACTOR_RETURN_XGBOOST/{version_name}")
print(f"Metrics: IC={mean_ic:.4f}, turnover={turnover:.6f}")

## Phase 4: Portfolio Construction

We use the ML-predicted returns to construct an optimal portfolio via mean-variance optimisation, then backtest against the benchmark. This is where the factor research becomes actionable: predicted returns from Phase 3 serve as the "expected return" input to the optimiser, which finds the portfolio weights that maximise risk-adjusted return subject to constraints (e.g., no shorting, position limits). The backtest compares cumulative performance, Sharpe ratio, and maximum drawdown against the SPX benchmark — validating whether the factor signals identified in Phases 2-3 translate into real portfolio alpha after accounting for diversification and risk.

In [ ]:
from scipy.optimize import minimize

latest_month = training_data["MONTH_DATE"].max()
latest = training_data[training_data["MONTH_DATE"] == latest_month].copy()
latest["PREDICTED_RETURN"] = final_model.predict(latest[feature_cols].fillna(0))

tickers = latest["TICKER"].values
n = len(tickers)
mu = latest["PREDICTED_RETURN"].values

returns_pivot = training_data.pivot_table(index="MONTH_DATE", columns="TICKER", values="FORWARD_RETURN").fillna(0)
returns_pivot = returns_pivot[[t for t in tickers if t in returns_pivot.columns]]
cov = returns_pivot.cov().values
valid_tickers = [t for t in tickers if t in returns_pivot.columns]
n = len(valid_tickers)
mu = latest.set_index("TICKER").loc[valid_tickers, "PREDICTED_RETURN"].values

def neg_sharpe(w):
    port_ret = w @ mu
    port_vol = np.sqrt(w @ cov @ w)
    return -port_ret / max(port_vol, 1e-10)

constraints = [{"type": "eq", "fun": lambda w: np.sum(w) - 1}]
bounds = [(0, 0.1)] * n
w0 = np.ones(n) / n

result = minimize(neg_sharpe, w0, method="SLSQP", bounds=bounds, constraints=constraints)
opt_weights = result.x

portfolio_df = pd.DataFrame({
    "TICKER": valid_tickers,
    "MONTH_DATE": latest_month,
    "WEIGHT": opt_weights,
    "EXPECTED_RETURN": mu,
    "RISK_CONTRIBUTION": opt_weights * (cov @ opt_weights) / max(np.sqrt(opt_weights @ cov @ opt_weights), 1e-10)
})

portfolio_sf = session.create_dataframe(portfolio_df)
portfolio_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.FACT_OPTIMAL_PORTFOLIO")

top_holdings = portfolio_df.nlargest(10, "WEIGHT")
print("Top 10 Holdings:")
print(top_holdings[["TICKER", "WEIGHT", "EXPECTED_RETURN"]].to_string(index=False))
print(f"\nPortfolio expected return: {(opt_weights @ mu):.4f}")
print(f"Portfolio volatility: {np.sqrt(opt_weights @ cov @ opt_weights):.4f}")
print(f"Sharpe ratio: {-result.fun:.4f}")

#### Portfolio Quality Gate: Implementability and Risk

A factor model with good IC can still produce an untradeable portfolio. These checks verify that the optimised portfolio is diversified, sector-balanced, and would have delivered reasonable risk-adjusted returns historically.

**What to look for:**
- **Weight concentration (Herfindahl index)** — HHI near 1/N means fully diversified; near 1.0 means a single-stock bet. For ~100 stocks with a 10% position cap, HHI should be 0.01-0.03.
- **Sector exposure** — Portfolio weight by sector should not be wildly skewed (e.g., 60% tech). Large sector tilts mean the model is learning sector momentum rather than stock-level alpha, and the portfolio is exposed to sector-specific drawdowns.
- **Backtest equity curve** — Cumulative return of the optimised portfolio vs benchmark over historical months. Consistent outperformance is more trustworthy than a single lucky period.
- **Maximum drawdown** — The worst peak-to-trough decline. Drawdowns > 30% may be unacceptable for institutional investors even if overall Sharpe is high.

**Decision impact:** If backtest Sharpe < 0.5, the strategy underperforms after typical transaction costs. If max drawdown > 30%, add risk constraints (e.g., sector limits, volatility targeting). If >40% of weight is concentrated in one sector, add sector diversification constraints to the optimiser. High HHI (>0.05) suggests the optimizer is exploiting estimation error in a few stocks — increase regularisation or tighten position caps.

In [ ]:
sector_pd = dim_sec.select("TICKER", "GICS_SECTOR_NAME").to_pandas()
port_with_sector = portfolio_df.merge(sector_pd, on="TICKER", how="left")
sector_weights = port_with_sector.groupby("GICS_SECTOR_NAME")["WEIGHT"].sum().sort_values(ascending=False)

months = sorted(training_data["MONTH_DATE"].unique())
port_returns = []
bench_returns = []
for i in range(len(months) - 1):
    m = months[i]
    m_data = training_data[training_data["MONTH_DATE"] == m].copy()
    m_data["PRED"] = final_model.predict(m_data[feature_cols].fillna(0))

    m_tickers = [t for t in m_data["TICKER"].values if t in returns_pivot.columns]
    if len(m_tickers) < 10:
        continue
    m_pred = m_data.set_index("TICKER").loc[m_tickers, "PRED"].values
    m_cov = returns_pivot[m_tickers].cov().values

    try:
        r = minimize(neg_sharpe.__class__(lambda w, mu=m_pred, cov=m_cov: -(w @ mu) / max(np.sqrt(w @ cov @ w), 1e-10)),
                     np.ones(len(m_tickers)) / len(m_tickers), method="SLSQP",
                     bounds=[(0, 0.1)] * len(m_tickers),
                     constraints=[{"type": "eq", "fun": lambda w: np.sum(w) - 1}])
        w = r.x
    except Exception:
        w = np.ones(len(m_tickers)) / len(m_tickers)

    next_month = months[i + 1]
    fwd = training_data[training_data["MONTH_DATE"] == next_month].set_index("TICKER")
    fwd_ret = fwd.reindex(m_tickers)["FORWARD_RETURN"].fillna(0).values
    port_returns.append(w @ fwd_ret)
    bench_returns.append(fwd_ret.mean())

port_cum = np.cumprod(1 + np.array(port_returns))
bench_cum = np.cumprod(1 + np.array(bench_returns))
drawdown = port_cum / np.maximum.accumulate(port_cum) - 1

hhi = (opt_weights ** 2).sum()

fig = make_subplots(rows=2, cols=2, subplot_titles=[
    "Portfolio Weight by Sector", f"Weight Concentration (HHI={hhi:.4f})",
    "Backtest Equity Curve", f"Drawdown (max = {drawdown.min()*100:.1f}%)"])

fig.add_trace(go.Bar(y=sector_weights.index, x=sector_weights.values, orientation="h",
    marker_color="#3498db", showlegend=False), row=1, col=1)
fig.add_vline(x=0.4, line_dash="dash", line_color="red", opacity=0.5, row=1, col=1)
fig.update_xaxes(title_text="Weight", row=1, col=1)

fig.add_trace(go.Bar(x=["Portfolio HHI", "Equal-Weight HHI"], y=[hhi, 1/n],
    marker_color=["#e67e22", "#2ecc71"], showlegend=False), row=1, col=2)
fig.update_yaxes(title_text="Herfindahl Index", row=1, col=2)

bt_dates = months[1:len(port_returns)+1]
fig.add_trace(go.Scatter(x=bt_dates, y=port_cum, mode="lines",
    name="Optimised Portfolio", line=dict(width=2)), row=2, col=1)
fig.add_trace(go.Scatter(x=bt_dates, y=bench_cum, mode="lines",
    name="Equal-Weight Benchmark", line=dict(width=1.5, dash="dash")), row=2, col=1)
fig.update_yaxes(title_text="Cumulative Return", row=2, col=1)

fig.add_trace(go.Scatter(x=bt_dates, y=drawdown, fill="tozeroy",
    fillcolor="rgba(255,0,0,0.3)", line=dict(color="red"), name="Drawdown", showlegend=False), row=2, col=2)
fig.add_hline(y=-0.3, line_dash="dash", line_color="black", opacity=0.3,
    annotation_text="-30% threshold", row=2, col=2)
fig.update_yaxes(title_text="Drawdown (%)", row=2, col=2)

fig.update_layout(height=650, template="plotly_white",
    title_text="Portfolio Construction Quality Checks", title_font_size=14,
    legend=dict(font_size=8))
fig.show()

port_sharpe = np.mean(port_returns) / max(np.std(port_returns), 1e-10) * np.sqrt(12)
max_dd = drawdown.min() * 100
max_sector = sector_weights.max()

print(f"\n{'Metric':<35} {'Value':>12} {'Status':>10}")
print("-" * 60)
print(f"{'HHI (concentration)':<35} {hhi:>12.4f} {'PASS' if hhi < 0.05 else 'REVIEW':>10}")
print(f"{'Max sector weight':<35} {max_sector:>11.1%} {'REVIEW' if max_sector > 0.4 else 'PASS':>10}")
print(f"{'Backtest Sharpe (annualised)':<35} {port_sharpe:>12.2f} {'PASS' if port_sharpe > 0.5 else 'REVIEW':>10}")
print(f"{'Max drawdown':<35} {max_dd:>11.1f}% {'FAIL' if max_dd < -30 else 'PASS':>10}")
print(f"{'# backtest months':<35} {len(port_returns):>12}")

## ML Observability and Pipeline Deployment

Once the model is live, we need to detect when factor relationships shift and automate the end-to-end workflow. Model Monitor tracks factor drift — if the distribution of input features changes materially (e.g., a volatility regime shift), the model's predictions may become unreliable and warrant retraining. The DAG API orchestrates the full monthly cycle: feature refresh, model scoring, and portfolio rebalance as a single managed pipeline. Together, these close the loop from research to production — ensuring the strategy adapts to changing markets rather than silently degrading.

In [ ]:
predictions_df = pd.DataFrame({
    "TICKER": training_data["TICKER"].values,
    "MONTH_DATE": training_data["MONTH_DATE"].values,
    "PREDICTED_RETURN": y_pred_all,
    "SHAP_TOP_FEATURE": [feature_cols[np.argmax(np.abs(shap_values[i]))] for i in range(len(shap_values))],
    "SHAP_TOP_VALUE": [float(shap_values[i, np.argmax(np.abs(shap_values[i]))]) for i in range(len(shap_values))],
    "MODEL_VERSION": version_name,
})

pred_sf = session.create_dataframe(predictions_df)
pred_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.FACT_ML_FACTOR_PREDICTIONS")
print(f"Predictions: {len(predictions_df)} rows written to FACT_ML_FACTOR_PREDICTIONS")

factor_scores_sf = session.create_dataframe(training_data[["TICKER", "MONTH_DATE"] + feature_cols])
factor_scores_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.FACT_FACTOR_SCORES")
print(f"Factor scores: {len(training_data)} rows written to FACT_FACTOR_SCORES")

In [ ]:
baseline_df = training_data[feature_cols + ["MONTH_DATE", "TICKER"]].dropna().copy()
baseline_df["PREDICTED_RETURN"] = y_pred_all
baseline_sf = session.create_dataframe(baseline_df)
baseline_sf.write.mode("overwrite").save_as_table(f"{DATABASE}.{ML_SCHEMA}.FACTOR_BASELINE")
print(f"Baseline table created: {DATABASE}.{ML_SCHEMA}.FACTOR_BASELINE ({len(baseline_df)} rows)")

monitor_sql = f"""
CREATE OR REPLACE MODEL MONITOR {DATABASE}.{ML_SCHEMA}.FACTOR_MODEL_MONITOR
WITH
    MODEL = {DATABASE}.{ML_SCHEMA}.FACTOR_RETURN_XGBOOST VERSION = '{version_name}'
    SOURCE = {DATABASE}.{ML_SCHEMA}.FACT_ML_FACTOR_PREDICTIONS
    WAREHOUSE = SAM_DEMO_EXECUTION_WH
    REFRESH_INTERVAL = '1 day'
    AGGREGATION_WINDOW = '30 days'
    TIMESTAMP_COLUMN = MONTH_DATE
    PREDICTION_SCORE_COLUMNS = (PREDICTED_RETURN)
    ID_COLUMNS = (TICKER)
    BASELINE = {DATABASE}.{ML_SCHEMA}.FACTOR_BASELINE
"""
session.sql(monitor_sql).collect()
print("Model Monitor created: FACTOR_MODEL_MONITOR (with baseline for drift detection)")

from snowflake.core import Root
from snowflake.core.task import Cron
from snowflake.core.task.dagv1 import DAG, DAGTask, DAGOperation

dag = DAG(
    name="FACTOR_WORKFLOW_PIPELINE",
    schedule=Cron("0 6 1 * *", "America/New_York"),
    warehouse="SAM_DEMO_EXECUTION_WH"
)

score_task = DAGTask(
    name="SCORE_FACTORS",
    definition=f"""
        INSERT INTO {DATABASE}.{ML_SCHEMA}.FACT_ML_FACTOR_PREDICTIONS
        WITH base_mkt AS (
            SELECT * FROM TABLE({DATABASE}.{ML_SCHEMA}.SECURITY_MARKET_FACTORS${version_name})
            WHERE MONTH_DATE = (SELECT MAX(MONTH_DATE) FROM TABLE({DATABASE}.{ML_SCHEMA}.SECURITY_MARKET_FACTORS${version_name}))
        ),
        base_fund AS (
            SELECT * FROM TABLE({DATABASE}.{ML_SCHEMA}.SECURITY_FUNDAMENTAL_FACTORS${version_name})
        ),
        base_sent AS (
            SELECT * FROM TABLE({DATABASE}.{ML_SCHEMA}.SECURITY_SENTIMENT_FACTORS${version_name})
        ),
        derived AS (
            SELECT * FROM TABLE({DATABASE}.{ML_SCHEMA}.SECURITY_DERIVED_FEATURES${version_name})
        ),
        base_hidden AS (
            SELECT * FROM TABLE({DATABASE}.{ML_SCHEMA}.SECURITY_HIDDEN_FACTORS${version_name})
        )
        SELECT
            m.TICKER, m.MONTH_DATE,
            MODEL({DATABASE}.{ML_SCHEMA}.FACTOR_RETURN_XGBOOST, '{version_name}')!predict(
                m.MOMENTUM_SCORE, f.VALUE_SCORE, f.QUALITY_SCORE, f.GROWTH_SCORE,
                f.SIZE_SCORE, m.VOLATILITY_SCORE, f.PROFITABILITY_SCORE, f.LEVERAGE_SCORE,
                f.EARNINGS_REVISION, f.DIVIDEND_YIELD_SCORE, m.BETA_SCORE, m.LIQUIDITY_SCORE,
                s.SENTIMENT_OVERALL, s.SENTIMENT_GUIDANCE, s.SENTIMENT_MARGINS,
                s.SENTIMENT_GROWTH, s.SENTIMENT_RISK,
                d.MOMENTUM_VALUE_RATIO, d.QUALITY_GROWTH_INTERACTION,
                d.SENTIMENT_MOMENTUM_INTERACTION, d.FACTOR_DISPERSION,
                COALESCE(h.AI_EXPOSURE_SCORE, 0) AS AI_EXPOSURE_SCORE,
                COALESCE(h.RESHORING_SCORE, 0) AS RESHORING_SCORE,
                COALESCE(h.RATE_CONVEXITY_SCORE, 0) AS RATE_CONVEXITY_SCORE,
                COALESCE(h.CLIMATE_SCORE, 0) AS CLIMATE_SCORE,
                COALESCE(h.GEOPOLITICAL_SCORE, 0) AS GEOPOLITICAL_SCORE
            ):output_feature_0::FLOAT AS PREDICTED_RETURN,
            NULL AS SHAP_TOP_FEATURE,
            NULL AS SHAP_TOP_VALUE,
            '{version_name}' AS MODEL_VERSION,
            CURRENT_TIMESTAMP() AS SCORED_AT
        FROM base_mkt m
        JOIN base_fund f ON m.SECURITYID = f.SECURITYID AND m.MONTH_DATE = f.MONTH_DATE
        LEFT JOIN base_sent s ON m.SECURITYID = s.SECURITYID AND m.MONTH_DATE = s.MONTH_DATE
        LEFT JOIN derived d ON m.SECURITYID = d.SECURITYID AND m.MONTH_DATE = d.MONTH_DATE
        LEFT JOIN base_hidden h ON m.SECURITYID = h.SECURITYID AND m.MONTH_DATE = h.MONTH_DATE
    """,
    warehouse="SAM_DEMO_EXECUTION_WH"
)
dag.add_task(score_task)

root = Root(session)
schema_ref = root.databases[DATABASE].schemas[ML_SCHEMA]
dag_op = DAGOperation(schema_ref)
dag_op.deploy(dag)
print(f"Pipeline deployed: FACTOR_WORKFLOW_PIPELINE (monthly, 1st at 06:00 ET)")

### From Research to Production

The hidden factor construction logic in Phase 1.8 above was the **origin story** — this is where we discovered, tested, and validated the 5 thematic factors. Once validated, the computation was productionised into the data build pipeline, which:

1. Runs the same z-score logic as a single SQL CTE chain (optimised for batch execution)
2. Aggregates to portfolio-level exposures (keyed by portfolio and date)
3. Feeds the Cortex Agent's hidden factor analyser tool via a semantic view

The notebook retains the security-level scores for ML training (via SECURITY_HIDDEN_FACTORS FeatureView), while the pipeline produces the portfolio-level view consumed by the agent and dashboards.

## Summary

This notebook demonstrated the complete quant factor workflow on a single Snowflake platform, using the full stock universe (~100 tickers) to ensure sufficient cross-sectional breadth:

| Phase | Capability | What We Used |
|---|---|---|
| **1. Factor Construction** | Feature Store | 3 domain-specific FeatureViews aligned with source cadences: SECURITY_MARKET_FACTORS (daily, 5 features), SECURITY_FUNDAMENTAL_FACTORS (60 days, 8 features), SECURITY_SENTIMENT_FACTORS (weekly, 5 features) |
| **1. Factor Construction** | UDF | Pre-aggregated group-by z-score normalisation (MIT — scale-invariant for trees) |
| **1.5 NLP Sentiment** | Cortex AI Functions | Snowpark `ai_sentiment()` with categories on real earnings transcripts — 5 sentiment dimensions |
| **1.6 Derived Features** | Feature Store | SECURITY_DERIVED_FEATURES FV — 4 interaction features (momentum/value, quality×growth, sentiment×momentum, dispersion) |
| **1.6 Forward Returns** | Feature Store | SECURITY_FORWARD_RETURNS FV — 21-day forward return as separate target (prevents train/serve skew) |
| **1.7 DT Health** | Feature Store | Dynamic Table refresh monitoring + staleness check |
| **1.8 Hidden Factors** | Feature Store + Snowpark | 5 hidden thematic factors (AI_Exposure, Reshoring, Rate_Convexity, Climate_Transition, Geopolitical_Risk) from SEC filings, NLP transcripts, ESG — z-scored with ±3σ winsorisation |
| **1.8 Validation** | Factor Quality Gates | Coverage (>50%), cross-sectional dispersion (std>0.3), univariate rank IC (>0.02), factor correlation matrix vs traditional |
| **2. Factor Model** | UDTF | `session.udtf.register()` — Fama-MacBeth validates which of the 11 factors (6 traditional + 5 hidden) earn significant premia |
| **3. ML Discovery** | Experiment Tracking | XGBoost + `SnowflakeXgboostCallback` — 27 features (13 quant + 5 NLP + 5 hidden + 4 derived), 3-way SHAP comparison |
| **3. ML Discovery** | SHAP | 3-way factor importance: Quant → +NLP → +Hidden, measuring incremental alpha from each data layer |
| **3. ML Discovery** | Leakage Validation | Timestamp-based check ensuring no future data in training set |
| **3. Model Registry** | Registry | `log_model()` with IC and turnover metrics |
| **4. Portfolio** | Optimisation | scipy.optimize mean-variance with ML-predicted returns |
| **Observability** | Model Monitor | `CREATE MODEL MONITOR` for factor drift |
| **Pipeline** | DAG API | Monthly factor refresh -> scoring -> rebalance |

This replaces 4 separate vendor tools (factor library, regression engine, ML platform, portfolio optimiser) with a single Snowflake workflow.